# Daily Land Surface Temperature Temporal Analysis

## What This Notebook Does

This notebook investigates **daily-scale Land Surface Temperature (LST) dynamics** in major Nigerian cities to understand **urban heat patterns, hot events, and persistent heat episodes** that impact public health and urban planning.

---

## Table of Contents

Navigate directly to any section:

1. [Setup & Configuration](#setup)
2. [Google Earth Engine Initialization](#gee-init)
3. [Load AOI Geometries](#load-aoi)
4. [Inter-Dataset Validation: MODIS vs ERA5-Land](#section-4)
   - [4a. Target Diagram: Multi-City Performance Summary](#section-4a)
5. [MODIS Data Extraction](#modis-extraction)
6. [ERA5-Land Data Extraction](#era5-extraction)
7. [SEVIRI Data Processing](#seviri-processing)
   - [7.1 Process SEVIRI LST](#seviri-process)
   - [7.2 Visualize SEVIRI Results](#seviri-viz)
8. [Cross-City Heat Analysis](#cross-city-analysis)
   - [7.1 Summary Chart: Mean LST Comparison](#summary-chart)
   - [7.2 Variability Chart: Temperature Stability](#variability-chart)
   - [7.3 Paired Dot Plot (Option A)](#combined-dot-plot)
   - [7.4 Day-Night Scatter with Pie Charts (Option C)](#scatter-pie)
9. [Summary & Next Steps](#summary)

---

## Sub-Research Objectives

1. **Quantify daily LST variability** across different sensor platforms (satellite observations vs. reanalysis models)
2. **Detect and characterize hot events** using statistical thresholds (90th percentile)
3. **Identify persistent heat episodes** lasting 2+ consecutive days that pose elevated health risks
4. **Validate inter-dataset consistency** between MODIS, ERA5-Land, and MSG SEVIRI
5. **Support stakeholder decision-making** with actionable heat event metrics


---

## Data Sources & Rationale

### Why Three Datasets?

Each dataset provides complementary perspectives on LST:

#### 1. MODIS Aqua (MYD11A1) - **Reference Standard**
- **Role**: High-quality satellite observations with established quality flags
- **Strengths**: 1km resolution, rigorous calibration, 20+ year record
- **Limitations**: Cloud contamination, 2 overpasses/day only, polar orbit gaps
- **Quality Control**: Hierarchical QA filtering (strict → moderate thresholds)

#### 2. ERA5-Land - **Gap-Filling Reanalysis**
- **Role**: Model-based estimates providing complete spatial/temporal coverage
- **Strengths**: No cloud gaps, hourly resolution, physically consistent
- **Limitations**: Coarser resolution (~9km), model biases, not direct observations
- **Processing**: Hourly data aggregated (median) to 3-hour windows matching MODIS

#### 3. MSG SEVIRI - **High-Frequency Monitoring**
- **Role**: Geostationary satellite with sub-hourly temporal resolution
- **Strengths**: 15-minute observations, continuous monitoring, ~3km resolution
- **Limitations**: EUMETSAT data access complexity, regional coverage, OpenDAP retrieval
- **Value**: Captures diurnal cycle evolution and rapid heat event onset

### Dataset Comparison Strategy

By analyzing **three independent datasets simultaneously**, we can:
- **Validate MODIS observations** against ERA5 reanalysis
- **Assess ERA5 model biases** using satellite truth
- **Evaluate SEVIRI consistency** with polar-orbiting sensors
- **Quantify uncertainty** through inter-dataset spread
- **Build robust conclusions** supported by multiple data sources

---

## Analytical Methods

### Hot Event Detection

**Definition**: A "hot day" is said to occur in our study when LST exceeds the 90th percentile of the annual distribution.


**Mathematical Formulation**:

Let $T = \{T_1, T_2, ..., T_n\}$ represent daily LST time series where $n = 365$ days.

$$P_{90} = \text{percentile}(T, 90)$$

$$H_i = \begin{cases} 
1 & \text{if } T_i \geq P_{90} \text{ (hot event)} \\
0 & \text{otherwise}
\end{cases}$$

### Persistent Event Identification

**Definition**: Consecutive hot days/nights lasting ≥2 days is said to constitute a "persistent heat episode."

**Rationale**:
- Single hot days may not trigger health impacts
- Multi-day heat exposure amplifies physiological stress
- Persistent events challenge urban infrastructure (cooling systems)
- Duration threshold captures meaningful heat waves

**Algorithm: Run-Length Encoding**

For the binary hot event sequence $H = \{H_1, H_2, ..., H_n\}$:

1. Identify consecutive sequences of $H_i = 1$
2. Calculate sequence duration $d$
3. Classify as persistent if $d \geq D_{\min}$ where $D_{\min} = 2$ days
4. Assign unique event IDs to persistent episodes

**Example**:
```
Day:       1  2  3  4  5  6  7  8  9  10
Hot:       0  1  1  1  0  0  1  1  0  1
Duration:  -  3  3  3  -  -  2  2  -  1
Event ID:  0  1  1  1  0  0  2  2  0  0
```
- Event 1: Days 2-4 (duration=3, persistent - YES)
- Event 2: Days 7-8 (duration=2, persistent - YES)
- Day 10: Isolated hot day (duration=1, not persistent - NO)

### Inter-Dataset Validation Metrics

To assess agreement between datasets $T_1$ (e.g., MODIS) and $T_2$ (e.g., ERA5):

**Pearson Correlation Coefficient ($r$)**:
$$r = \frac{\sum_{i=1}^{n}(T_{1i} - \bar{T}_1)(T_{2i} - \bar{T}_2)}{\sqrt{\sum_{i=1}^{n}(T_{1i} - \bar{T}_1)^2 \sum_{i=1}^{n}(T_{2i} - \bar{T}_2)^2}}$$

Interpretation: $r > 0.9$ = excellent, $0.7-0.9$ = good, $< 0.7$ = poor agreement

**Bias (Mean Difference)**:
$$\text{Bias} = \frac{1}{n}\sum_{i=1}^{n}(T_{1i} - T_{2i})$$

Interpretation: Positive = $T_1$ warmer than $T_2$, reveals systematic offsets

**Root Mean Square Error (RMSE)**:
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(T_{1i} - T_{2i})^2}$$

Interpretation: Overall prediction error magnitude (°C units)

**Mean Absolute Error (MAE)**:
$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|T_{1i} - T_{2i}|$$

Interpretation: Robust to outliers, intuitive error metric

---

## Notebook Structure

This notebook follows a **modular, reproducible workflow**:

### Processing Pipeline

```
┌─────────────────────────────────────────────────────────────┐
│ SECTION 1: Setup & Initialization                          │
│ • Load libraries and configuration                          │
│ • Initialize Google Earth Engine                            │
│ • Load EUMETSAT credentials                                 │
│ • Select city and load AOI geometry                         │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 2: Data Extraction                                 │
│ • Extract MODIS Aqua LST (day/night) with QA filtering     │
│ • Extract ERA5-Land LST with hourly aggregation            │
│ • Extract MSG SEVIRI LST via OpenDAP (optional)            │
│ • Implement skip logic for existing files                  │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 3: Hot Event Analysis                              │
│ • Calculate 90th percentile thresholds                      │
│ • Detect hot events for each dataset                        │
│ • Identify persistent heat episodes (≥2 days)              │
│ • Export event DataFrames and statistics                    │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 4: Inter-Dataset Comparison                        │
│ • Compare MODIS vs ERA5-Land                                │
│ • Compare MODIS vs SEVIRI (if available)                    │
│ • Compare ERA5 vs SEVIRI (if available)                     │
│ • Calculate correlation, bias, RMSE, MAE                    │
│ • Create Target Diagram for multi-city summary             │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 5: Visualization                                   │
│ • Time series plots with hot event highlighting            │
│ • Persistent event duration annotations                     │
│ • Dataset comparison scatter plots                          │
│ • Diurnal range analysis (day-night difference)            │
│ • Comprehensive multi-panel dashboards                      │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 6: Export & Documentation                          │
│ • Export CSV time series (by dataset/time)                 │
│ • Export JSON summary statistics                            │
│ • Save visualizations (PNG + PDF)                           │
│ • Generate comprehensive analysis summary                   │
└─────────────────────────────────────────────────────────────┘
```

### Code Organization

All heavy processing is **modularized into reusable utilities** (`src/` directory):

- **`gee_utils.py`**: Google Earth Engine initialization, MODIS/ERA5 extraction
- **`seviri_utils.py`**: EUMETSAT credentials, SEVIRI OpenDAP retrieval
- **`temporal_analysis.py`**: Hot event detection, persistent event identification, comparison metrics
- **`visualization.py`**: Time series plots, comparison dashboards, multi-panel figures, target diagrams, paired dot plot (`create_paired_dot_plot`), day-night scatter with pie charts (`create_daynight_scatter_pie`)
- **`data_io.py`**: Standardized CSV/JSON exports with consistent naming


## Output Structure

All results are saved to: `data/outputs/04_trend_analysis_daily/City_{city_name}/`

```
City_Lagos/
├── Data/
│   ├── modis_day_2024.csv              # MODIS daytime LST time series
│   ├── modis_night_2024.csv            # MODIS nighttime LST time series
│   ├── modis_day_events_2024.csv       # Hot events detected (day)
│   ├── modis_night_events_2024.csv     # Hot events detected (night)
│   ├── modis_day_stats_2024.json       # Summary statistics (day)
│   ├── modis_night_stats_2024.json     # Summary statistics (night)
│   ├── era5_day_2024.csv               # ERA5-Land daytime LST
│   ├── era5_night_2024.csv             # ERA5-Land nighttime LST
│   ├── seviri_day_2024.csv             # SEVIRI daytime LST (if extracted)
│   ├── seviri_night_2024.csv           # SEVIRI nighttime LST (if extracted)
│   ├── comparison_modis_vs_era5_2024.csv      # Merged comparison data
│   ├── comparison_modis_vs_era5_stats_2024.json  # Comparison metrics
│   └── comprehensive_summary_2024.json  # Full analysis summary
│
└── Figures/
    ├── modis_day_timeseries.png/pdf    # Time series visualizations
    ├── modis_day_persistent_events.png/pdf
    ├── comparison_modis_vs_era5.png/pdf
    ├── target_diagram_MODIS_vs_ERA5-Land.png/pdf  # Target diagram
    ├── modis_diurnal_range.png/pdf
    └── comprehensive_daily_dashboard.png/pdf  # Multi-panel dashboard
```

---

## Configuration

Analysis parameters are centralized in **`config.py`**:

```python
DAILY_LST_CONFIG = {
    'hot_percentile': 90,          # Hot event threshold
    'persistent_threshold': 2,     # Minimum consecutive days
    'skip_existing': True,         # Skip if files exist
    'study_year': 2024
}
```

Modify `config.py` to adjust thresholds without changing notebook code.

---

<a id="setup"></a>
## Setup & Configuration

Load project libraries, configuration, and define analysis parameters.

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
import sys
import warnings
from pathlib import Path
import importlib

# Add project root and src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))  # For config.py
sys.path.insert(0, str(project_root / "src"))  # For src modules

# Data manipulation
import numpy as np
import pandas as pd
import geopandas as gpd
import ee

# Visualization
import matplotlib.pyplot as plt

# Parallelism for GEE extraction (ThreadPoolExecutor)
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Progress bars for heavy-compute loops
from tqdm.notebook import tqdm

# Project modules (with reload to ensure latest versions)
import config
importlib.reload(config)

import gee_utils
importlib.reload(gee_utils)
from gee_utils import (initialize_gee, extract_modis_daily_lst,
                        extract_era5_daily_lst, validate_extraction_metadata)

import temporal_analysis
importlib.reload(temporal_analysis)
from temporal_analysis import (detect_hot_events, identify_persistent_events, 
                                compare_datasets, summarize_daily_lst_analysis)

import data_io
importlib.reload(data_io)
from data_io import (export_daily_lst_data, export_daily_lst_events, 
                     export_dataset_comparison, export_comprehensive_daily_summary)

import visualization
importlib.reload(visualization)
from visualization import (plot_daily_lst_timeseries, plot_persistent_events, 
                           plot_dataset_comparison, plot_diurnal_range, 
                           create_comprehensive_daily_dashboard,
                           create_persistent_heat_event_maps,
                           create_persistent_heat_event_summary_chart,
                           create_persistent_heat_event_variability_chart,
                           create_paired_dot_plot,
                           create_daynight_scatter_pie)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Configuration from config.py
STUDY_YEAR = config.DAILY_LST_CONFIG['study_year']
HOT_PERCENTILE = config.DAILY_LST_CONFIG['hot_percentile']
PERSISTENT_THRESHOLD = config.DAILY_LST_CONFIG['persistent_threshold']
SKIP_EXISTING = config.DAILY_LST_CONFIG['skip_existing']

# Parallelism: number of concurrent GEE extraction workers
# Conservative default (3) stays well within GEE's ~20-40 concurrent request quota.
# Increase to 4-6 for faster runs if your GEE project is not shared.
MAX_GEE_WORKERS = 3

# ERA5-Land time windows (UTC hours)
ERA5_NIGHT_HOURS = [0, 1, 2]  # 00:00, 01:00, 02:00 UTC (aligns with MODIS night ~1:30 AM)
ERA5_DAY_HOURS = [12, 13, 14]  # 12:00, 13:00, 14:00 UTC (aligns with MODIS day ~1:30 PM)

# SEVIRI processing config
SEVIRI_CONFIG = {
    'temporal_strategy': '15min',  # '15min', '1hr', or 'daily'
    'day_window': (12, 15),  # 12:00-15:00 UTC
    'night_window': (0, 3)   # 00:00-03:00 UTC  
}

# Base paths
aoi_base_dir = project_root / 'data' / 'outputs' / '01_aoi_delineation'

# Auto-discover cities from AOI directory
aoi_dir = aoi_base_dir
CITIES = sorted([
    d.name.replace('City_', '') 
    for d in aoi_base_dir.glob('City_*') 
    if d.is_dir()
])

if not CITIES:
    raise FileNotFoundError(
        f"No cities found in {aoi_base_dir}\n"
        f"Please run Notebook 01 (AOI Delineation) first to create city boundaries."
    )

# Date range for study year
DATE_RANGE = pd.date_range(f'{STUDY_YEAR}-01-01', f'{STUDY_YEAR}-12-31', freq='D')

# Display configuration
print("="*80)
print("DAILY LST TEMPORAL ANALYSIS - CONFIGURATION")
print("="*80)
print(f"Study Year:           {STUDY_YEAR}")
print(f"Date Range:           {DATE_RANGE[0].date()} to {DATE_RANGE[-1].date()} ({len(DATE_RANGE)} days)")
print(f"Hot Percentile:       {HOT_PERCENTILE}th percentile")
print(f"Persistent Threshold: {PERSISTENT_THRESHOLD} consecutive days")
print(f"Skip Existing Files:  {SKIP_EXISTING}")
print(f"GEE Parallel Workers: {MAX_GEE_WORKERS}")
print(f"\nExtraction Methods:")
print(f"  MODIS: {config.EXTRACTION_METHODS['modis']['version']} "
      f"(reducer={config.EXTRACTION_METHODS['modis']['spatial_reducer']})")
print(f"  ERA5:  {config.EXTRACTION_METHODS['era5']['version']} "
      f"(reducer={config.EXTRACTION_METHODS['era5']['spatial_reducer']})")
print(f"\nERA5-Land Time Windows (UTC):")
print(f"  Night: {ERA5_NIGHT_HOURS} (aligns with MODIS 1:30 AM)")
print(f"  Day:   {ERA5_DAY_HOURS} (aligns with MODIS 1:30 PM)")
print(f"\nCities Discovered: {', '.join(CITIES)} ({len(CITIES)} total)")
print(f"  Source: {aoi_base_dir}")
print("="*80)
print("Configuration loaded successfully")

<a id="gee-init"></a>
## 🌍 Initialize Google Earth Engine

Connect to Google Earth Engine using service account credentials for data access.

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# Initialize Google Earth Engine
print("="*80)
print("INITIALIZING AUTHENTICATION")
print("="*80)

# GEE Initialization
print("\n[1/2] Initializing Google Earth Engine...")
gee_success = initialize_gee(
    service_account_key=config.GEE_CONFIG['service_json'],
    project_id=config.GEE_CONFIG['project_id']
)

if not gee_success:
    raise RuntimeError("GEE initialization failed. Check service account credentials.")

print("    GEE authenticated successfully")

# Load EUMETSAT credentials
print("\n[2/2] Loading EUMETSAT credentials...")
try:
    import json
    credentials_file = config.EUMETSAT_CONFIG['credentials_file']
    with open(credentials_file, 'r') as f:
        eumetsat_creds = json.load(f)
    
    # Validate required keys
    if 'consumer_key' not in eumetsat_creds or 'consumer_secret' not in eumetsat_creds:
        raise ValueError("Missing required keys: consumer_key or consumer_secret")
    
    print(f"    EUMETSAT credentials loaded")
    print(f"    Consumer Key: {eumetsat_creds['consumer_key'][:20]}...")
    EUMETSAT_AVAILABLE = True
except Exception as e:
    print(f"    EUMETSAT credentials not available: {e}")
    print(f"    SEVIRI extraction will be skipped.")
    EUMETSAT_AVAILABLE = False
    eumetsat_creds = None

print("="*80)
print("Authentication initialization complete")
print("="*80)

<a id="load-aoi"></a>
## 🗺️ Load Area of Interest (AOI) Geometries

**Purpose**: Load city boundaries for all Nigerian cities in the study

**What This Section Does**:
- Auto-discovers processed cities from Notebook 01 output directory
- Loads GeoJSON boundaries for each city
- Validates geometry completeness
- Converts to Google Earth Engine format for spatial filtering

**Expected Output**: Dictionary of city geometries ready for LST extraction

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ====================================================================
# DISCOVER AND LOAD ALL CITY AOIS
# This cell automatically loads all city AOIs for processing
# No hardcoding needed - fully reproducible across different users
# ====================================================================

print("="*80)
print("DISCOVERING AND LOADING CITY AOIS")
print("="*80)

# Dictionary to store city geometries and metadata
city_geometries = {}
city_metadata = {}

# Load AOI for each discovered city
for city_name in tqdm(CITIES, desc="Loading AOIs", unit="city"):
    # Construct AOI file path
    aoi_dir = config.OUTPUT_DIR / "01_aoi_delineation" / f"City_{city_name}" / "Features"
    aoi_file = aoi_dir / f"{city_name}_AOI_Boundary.geojson"
    
    # Check if file exists
    if not aoi_file.exists():
        tqdm.write(f"Skipping {city_name}: AOI file not found at {aoi_file}")
        continue
    
    try:
        # Load AOI GeoDataFrame
        aoi_gdf = gpd.read_file(aoi_file)
        
        # Extract bounds (used for metadata/display)
        bounds = aoi_gdf.total_bounds  # [minx, miny, maxx, maxy]
        bbox = (bounds[0], bounds[1], bounds[2], bounds[3])  # (min_lon, min_lat, max_lon, max_lat)
        
        # Convert actual polygon geometry to Earth Engine Geometry
        # Uses the real AOI shape (may be clipped to land boundary for coastal cities)
        # instead of just a bounding box rectangle which could include ocean
        geom = aoi_gdf.geometry.unary_union
        aoi_ee_geometry = ee.Geometry(geom.__geo_interface__)
        
        # Calculate area in square kilometers (project to UTM for accurate calculation)
        # Use UTM Zone 32N (EPSG:32632) which covers Nigeria
        aoi_projected = aoi_gdf.to_crs('EPSG:32632')
        area_sq_km = aoi_projected.geometry.area.sum() / 1_000_000  # Convert m² to km²
        
        # Store in dictionary
        city_geometries[city_name] = {
            'gdf': aoi_gdf,
            'ee_geometry': aoi_ee_geometry,
            'bbox': bbox,
            'bounds': bounds
        }
        
        # Store metadata
        city_metadata[city_name] = {
            'crs': str(aoi_gdf.crs),
            'features': len(aoi_gdf),
            'area_sq_km': area_sq_km,
            'lon_range': (bbox[0], bbox[2]),
            'lat_range': (bbox[1], bbox[3])
        }
        
        tqdm.write(f"{city_name:15s} | Area: {area_sq_km:8.2f} km² | "
              f"Bounds: [{bbox[0]:.2f}, {bbox[1]:.2f}, {bbox[2]:.2f}, {bbox[3]:.2f}]")
        
    except Exception as e:
        tqdm.write(f"Error loading {city_name}: {e}")
        continue

# Summary
print("\n" + "="*80)
print(f"Successfully loaded {len(city_geometries)} city AOIs:")
for i, city_name in enumerate(city_geometries.keys(), 1):
    print(f"   {i:2d}. {city_name}")

if len(city_geometries) == 0:
    raise RuntimeError(
        "No city AOIs loaded successfully!\n"
        "Please run Notebook 01 (AOI Delineation) first to create city boundaries."
    )

print(f"\nReady for data extraction from MODIS, ERA5-Land, and SEVIRI")
print(f"Next: Run SECTION 2 to extract data (will skip existing files automatically)")  
print("="*80)

---

## SECTION 2: Data Extraction

Extract daily LST data from three complementary datasets for all discovered cities.

### Extraction Strategy

For each city, we extract:
1. **MODIS Aqua LST** (day + night with hierarchical QA filtering)
2. **ERA5-Land LST** (day + night aggregated from hourly data)
3. **MSG SEVIRI LST** (day + night via OpenDAP - optional)

### Smart Skip Logic (3-tier validation)

When `skip_existing = True`, cached files go through **three checks** before being accepted:

| Check | What it validates | Action on failure |
|-------|-------------------|-------------------|
| **1. Method version** | `.meta.json` sidecar matches `config.EXTRACTION_METHODS[dataset]['version']` | Delete cache + re-extract |
| **2. Spatial reducer** | Reducer type (e.g. `median` vs `mean`) matches current config | Delete cache + re-extract |
| **3. Data quality** | At least one row has non-NaN LST values | Delete cache + re-extract |

This means you can safely **Restart Kernel → Run All** after changing the extraction methodology (e.g. switching ERA5 from `mean()` to `median()`). Stale caches are automatically invalidated.

### Output Files

Results saved to: `data/outputs/04_trend_analysis_daily/City_{city}/Data/`
- `{city}_raw_daily_MODIS_{year}.csv` + `.meta.json`
- `{city}_raw_daily_ERA5_{year}.csv` + `.meta.json`
- `seviri_day_{year}.csv`, `seviri_night_{year}.csv` (if EUMETSAT available)

---

### 4. MODIS Aqua LST Extraction

Extract daily MODIS Aqua LST using the **MYD11A1 product** (Collection 6.1).

**Product Details:**
- **Temporal Coverage**: 2 overpasses per day
  - Night: ~1:30 AM local time
  - Day: ~1:30 PM local time
- **Spatial Resolution**: 1km native resolution
- **Quality Control**: Hierarchical QA filtering
  - Strict thresholds: LST error ≤ 2K, Good quality flags
  - Moderate thresholds: LST error ≤ 3K, if strict fails
  - Filters invalid values (<5°C or >70°C)

**Processing:**
1. Loop through all discovered cities
2. Extract day LST for each city (366 days in 2024)
3. Apply hierarchical QA filtering
4. Export to CSV with standardized naming

In [ ]:
# ====================================================================
# EXTRACT MODIS AQUA LST (DAY + NIGHT) FOR ALL CITIES
# Parallel city-level extraction via ThreadPoolExecutor
# Server-side batch GEE calls (~180× fewer API round-trips per city)
# ====================================================================

print("="*80)
print("EXTRACTING MODIS AQUA LST (DAY + NIGHT)")
print(f"Current method: {config.EXTRACTION_METHODS['modis']['version']} "
      f"(reducer={config.EXTRACTION_METHODS['modis']['spatial_reducer']})")
print(f"Parallel workers: {MAX_GEE_WORKERS}")
print("="*80)

modis_results = {}
_print_lock = threading.Lock()

# ── Phase 1: Skip logic — identify which cities need extraction ──────
cities_to_extract = {}  # city_name -> (city_geom, output_file)

for city_name in tqdm(list(city_geometries.keys()), desc="MODIS skip check", unit="city"):
    city_geom = city_geometries[city_name]
    city_output_dir = config.OUTPUT_DIR / "04_trend_analysis_daily" / f"City_{city_name}" / "Data"
    city_output_dir.mkdir(parents=True, exist_ok=True)
    output_file = city_output_dir / f"{city_name}_raw_daily_MODIS_{STUDY_YEAR}.csv"
    
    needs_extraction = True
    if SKIP_EXISTING and output_file.exists():
        meta_ok, meta_reason = validate_extraction_metadata(output_file, 'modis')
        if not meta_ok:
            tqdm.write(f"   ⚠️  {city_name}: Method change detected: {meta_reason}")
            tqdm.write(f"   🔄 Deleting stale cache and re-extracting...")
            output_file.unlink()
            meta_path = output_file.with_suffix('.meta.json')
            if meta_path.exists():
                meta_path.unlink()
        else:
            cached_df = pd.read_csv(output_file)
            lst_cols = [c for c in cached_df.columns if 'lst_' in c]
            valid_count = cached_df[lst_cols].notna().any(axis=1).sum()
            
            if valid_count > 0:
                modis_results[city_name] = cached_df
                valid_pct = valid_count / len(cached_df) * 100
                tqdm.write(f"   ✅ [SKIP] {city_name}: Loaded {len(cached_df)} records "
                      f"({valid_count} valid days, {valid_pct:.1f}%)")
                needs_extraction = False
            else:
                tqdm.write(f"   ⚠️  {city_name}: Cached file has ALL NaN values - deleting and re-extracting...")
                output_file.unlink()
                meta_path = output_file.with_suffix('.meta.json')
                if meta_path.exists():
                    meta_path.unlink()
    
    if needs_extraction:
        cities_to_extract[city_name] = (city_geom, output_file)

# ── Phase 2: Parallel extraction of non-cached cities ────────────────
if cities_to_extract:
    print(f"\n{'='*80}")
    print(f"PARALLEL GEE EXTRACTION: {len(cities_to_extract)} cities × "
          f"{MAX_GEE_WORKERS} workers")
    print(f"{'='*80}")
    
    def _extract_modis_worker(city_name, city_geom, output_file):
        """Worker function for parallel MODIS extraction."""
        try:
            success = extract_modis_daily_lst(
                geometry=city_geom['ee_geometry'],
                city_name=city_name,
                year=STUDY_YEAR,
                output_path=output_file
            )
            if success:
                df = pd.read_csv(output_file)
                return city_name, df, None
            else:
                return city_name, None, "Insufficient data extracted"
        except Exception as e:
            return city_name, None, str(e)
    
    with ThreadPoolExecutor(max_workers=MAX_GEE_WORKERS) as executor:
        futures = {
            executor.submit(_extract_modis_worker, name, geom, path): name
            for name, (geom, path) in cities_to_extract.items()
        }
        
        pbar = tqdm(as_completed(futures), total=len(futures),
                     desc="MODIS extraction", unit="city")
        for future in pbar:
            city_name, df, error = future.result()
            with _print_lock:
                if df is not None:
                    modis_results[city_name] = df
                    tqdm.write(f"   ✅ {city_name}: "
                          f"{len(df)} records | "
                          f"Day: {df['day_lst_median'].min():.1f}–{df['day_lst_median'].max():.1f}°C | "
                          f"Night: {df['night_lst_median'].min():.1f}–{df['night_lst_median'].max():.1f}°C")
                else:
                    tqdm.write(f"   ❌ {city_name}: {error}")
else:
    print(f"\n   All {len(city_geometries)} cities loaded from cache — no extraction needed")

# ── Phase 3: Summary ─────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"MODIS EXTRACTION COMPLETE: {len(modis_results)}/{len(city_geometries)} cities processed")
print(f"{'='*80}")

### 5. ERA5-Land LST Extraction

Extract daily ERA5-Land LST using the **ERA5-Land hourly reanalysis** product.

**Product Details:**
- **Resolution:** ~9 km (0.1° grid)  
- **Temporal Coverage:** Hourly data  
- **Provider:** ECMWF via Google Earth Engine  

**ERA5-Land Configuration:**
- **Daytime Hours:** 13:00, 14:00, 15:00 UTC (afternoon peak heating, aligns with MODIS Aqua ~1:30 PM)
- **Nighttime Hours:** 01:00, 02:00, 03:00 UTC (early morning minimum, aligns with MODIS Aqua ~1:30 AM)
- **Aggregation:** Median across 3-hour window for each day

**Processing Workflow:**
1. Loop through all discovered cities
2. Extract hourly LST for specified time windows (day/night)
3. Aggregate to daily median LST for each time period
4. Combine day+night into single CSV
5. Export to: `{city}_raw_daily_ERA5_{year}.csv`

**Advantages:**
- Gap-free temporal coverage (no missing days)
- Consistent spatial resolution across all locations
- Physics-based reanalysis with assimilated observations

**Limitations:**
- Coarser spatial resolution (~9 km vs. 1 km for MODIS)
- Reanalysis estimates (not direct observations)
- Time lag in data availability (~5 days)

In [ ]:
# ====================================================================
# EXTRACT ERA5-LAND LST (DAY + NIGHT) FOR ALL CITIES
# Parallel city-level extraction via ThreadPoolExecutor
# ERA5 already uses server-side map() + single getInfo() per city
# ====================================================================

print("="*80)
print("EXTRACTING ERA5-LAND LST (DAY + NIGHT)")
print(f"Current method: {config.EXTRACTION_METHODS['era5']['version']} "
      f"(reducer={config.EXTRACTION_METHODS['era5']['spatial_reducer']})")
print(f"Parallel workers: {MAX_GEE_WORKERS}")
print("="*80)

era5_results = {}
_print_lock_era5 = threading.Lock()

# ── Phase 1: Skip logic — identify which cities need extraction ──────
cities_to_extract_era5 = {}  # city_name -> (city_geom, output_file)

for city_name in tqdm(list(city_geometries.keys()), desc="ERA5 skip check", unit="city"):
    city_geom = city_geometries[city_name]
    city_output_dir = config.OUTPUT_DIR / "04_trend_analysis_daily" / f"City_{city_name}" / "Data"
    city_output_dir.mkdir(parents=True, exist_ok=True)
    output_file = city_output_dir / f"{city_name}_raw_daily_ERA5_{STUDY_YEAR}.csv"
    
    needs_extraction = True
    if SKIP_EXISTING and output_file.exists():
        meta_ok, meta_reason = validate_extraction_metadata(output_file, 'era5')
        if not meta_ok:
            tqdm.write(f"   ⚠️  {city_name}: Method change detected: {meta_reason}")
            tqdm.write(f"   🔄 Deleting stale cache and re-extracting...")
            output_file.unlink()
            meta_path = output_file.with_suffix('.meta.json')
            if meta_path.exists():
                meta_path.unlink()
        else:
            cached_df = pd.read_csv(output_file)
            lst_cols = [c for c in cached_df.columns if 'lst_' in c]
            valid_count = cached_df[lst_cols].notna().any(axis=1).sum()
            
            if valid_count > 0:
                era5_results[city_name] = cached_df
                valid_pct = valid_count / len(cached_df) * 100
                tqdm.write(f"   ✅ [SKIP] {city_name}: Loaded {len(cached_df)} records "
                      f"({valid_count} valid days, {valid_pct:.1f}%)")
                needs_extraction = False
            else:
                tqdm.write(f"   ⚠️  {city_name}: Cached file has ALL NaN values - deleting and re-extracting...")
                output_file.unlink()
                meta_path = output_file.with_suffix('.meta.json')
                if meta_path.exists():
                    meta_path.unlink()
    
    if needs_extraction:
        cities_to_extract_era5[city_name] = (city_geom, output_file)

# ── Phase 2: Parallel extraction of non-cached cities ────────────────
if cities_to_extract_era5:
    print(f"\n{'='*80}")
    print(f"PARALLEL GEE EXTRACTION: {len(cities_to_extract_era5)} cities × "
          f"{MAX_GEE_WORKERS} workers")
    print(f"{'='*80}")
    
    def _extract_era5_worker(city_name, city_geom, output_file):
        """Worker function for parallel ERA5 extraction."""
        try:
            success = extract_era5_daily_lst(
                geometry=city_geom['ee_geometry'],
                city_name=city_name,
                year=STUDY_YEAR,
                output_path=output_file
            )
            if success:
                df = pd.read_csv(output_file)
                return city_name, df, None
            else:
                return city_name, None, "Insufficient data extracted"
        except Exception as e:
            return city_name, None, str(e)
    
    with ThreadPoolExecutor(max_workers=MAX_GEE_WORKERS) as executor:
        futures = {
            executor.submit(_extract_era5_worker, name, geom, path): name
            for name, (geom, path) in cities_to_extract_era5.items()
        }
        
        pbar = tqdm(as_completed(futures), total=len(futures),
                     desc="ERA5 extraction", unit="city")
        for future in pbar:
            city_name, df, error = future.result()
            with _print_lock_era5:
                if df is not None:
                    era5_results[city_name] = df
                    tqdm.write(f"   ✅ {city_name}: "
                          f"{len(df)} records | "
                          f"Day: {df['day_lst_median'].min():.1f}–{df['day_lst_median'].max():.1f}°C | "
                          f"Night: {df['night_lst_median'].min():.1f}–{df['night_lst_median'].max():.1f}°C")
                else:
                    tqdm.write(f"   ❌ {city_name}: {error}")
else:
    print(f"\n   All {len(city_geometries)} cities loaded from cache — no extraction needed")

# ── Phase 3: Summary ─────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"ERA5-LAND EXTRACTION COMPLETE: {len(era5_results)}/{len(city_geometries)} cities processed")
print(f"{'='*80}")

---

## SECTION 3: Hot Event Analysis & Statistics

Analyze extracted LST data to identify and characterize extreme heat events.

### Analysis Workflow

For each city with extracted MODIS and ERA5-Land data, we:

1. **Load Data**: Load both datasets from CSV files
2. **Calculate Thresholds**: Compute 90th percentile thresholds for hot event detection
3. **Identify Hot Events**: Flag days/nights exceeding the threshold
4. **Find Persistent Events**: Detect consecutive hot days (≥2 days duration)
5. **Generate Statistics**: Calculate comprehensive temperature and event metrics
6. **Create Visualizations**: Produce dual-dataset comparison plots
7. **Export Results**: Save JSON statistics and PNG visualizations

### Hot Event Metrics

For each dataset (MODIS and ERA5-Land), we calculate:

**Data Coverage:**
- Total days with valid observations
- Day/night coverage percentages

**Temperature Statistics:**
- Min, median, mean, max, std for each LST metric (day/night, min/median/mean/max)

**Heat Event Detection:**
- 90th percentile thresholds (day and night)
- Count of hot days and hot nights
- Persistent hot events (≥2 consecutive days)
- Longest persistent period across both datasets

**Extreme Events:**
- Highest day/night temperatures
- Dates of extreme events

### Output Files

For each city:
- **JSON**: `{city}_comprehensive_LST_statistics_{year}.json` - Full statistics
- **PNG**: `{city}_MODIS_vs_ERA5Land_comparison_{year}.png` - Visualization

---

In [ ]:
# ====================================================================
# HOT EVENT ANALYSIS FOR ALL CITIES (MODIS + ERA5-LAND)
# ====================================================================

# Import additional required modules
from temporal_analysis_utils import (
    load_dual_datasets, 
    calculate_comprehensive_heat_statistics_dual,
    save_comprehensive_statistics_json
)

# Reload visualization to ensure latest version with stats parameter
import visualization

print("="*80)
print("HOT EVENT ANALYSIS (MODIS + ERA5-LAND)")
print("="*80)

analysis_results = []

for city_name in tqdm(list(city_geometries.keys()), desc="Hot event analysis", unit="city"):
    tqdm.write(f"\n{'='*80}")
    tqdm.write(f"Processing: {city_name}")
    tqdm.write(f"{'='*80}")
    
    # Define output paths
    city_output_dir = config.OUTPUT_DIR / "04_trend_analysis_daily" / f"City_{city_name}" / "Data"
    city_plots_dir = config.OUTPUT_DIR / "04_trend_analysis_daily" / f"City_{city_name}" / "Figures"
    city_plots_dir.mkdir(parents=True, exist_ok=True)
    
    # Check for existing analysis files (figures always regenerated)
    stats_file = city_output_dir / f"{city_name}_comprehensive_LST_statistics_{STUDY_YEAR}.json"
    plot_file = city_plots_dir / f"{city_name}_MODIS_vs_ERA5Land_comparison_{STUDY_YEAR}.png"
    
    try:
        # Load datasets
        tqdm.write(f"    Loading datasets...")
        modis_df, era5_df, data_status = load_dual_datasets(
            city_name, STUDY_YEAR, config.OUTPUT_DIR
        )
        
        if data_status == 'no_data':
            tqdm.write(f"{city_name}: No data available")
            analysis_results.append({
                'city': city_name,
                'status': 'no_data'
            })
            continue
        
        # Determine which datasets are currently available
        current_datasets = set()
        if modis_df is not None and len(modis_df) > 0:
            current_datasets.add('MODIS')
        if era5_df is not None and len(era5_df) > 0:
            current_datasets.add('ERA5-Land')
        
        # Check if stats already exist and should skip recalculation
        # Also validate that the cached stats cover the same datasets as currently available
        skip_stats = False
        if SKIP_EXISTING and stats_file.exists():
            import json
            with open(stats_file, 'r') as f:
                cached_stats = json.load(f)
            cached_datasets = set(cached_stats.get('datasets_available', []))
            
            if cached_datasets == current_datasets:
                stats = cached_stats
                tqdm.write(f"    Loading existing statistics from JSON...")
                tqdm.write(f"        Loaded statistics: {stats_file.name} (datasets: {cached_datasets})")
                skip_stats = True
            else:
                tqdm.write(f"    STALE: Cached stats have {cached_datasets} but current data has {current_datasets}")
                tqdm.write(f"    Recomputing statistics...")
        
        if not skip_stats:
            # Calculate comprehensive statistics
            tqdm.write(f"    Analyzing heat events...")
            stats = calculate_comprehensive_heat_statistics_dual(
                modis_df, era5_df, city_name, HOT_PERCENTILE, PERSISTENT_THRESHOLD
            )
            
            # Save statistics JSON
            save_comprehensive_statistics_json(stats, stats_file)
            tqdm.write(f"        Saved statistics: {stats_file.name}")
        
        # Create visualization with hot event annotations (always regenerated)
        tqdm.write(f"    Creating visualization...")
        plot_path = visualization.create_dual_dataset_comparison_plot(
            modis_df, era5_df, city_name, STUDY_YEAR, city_plots_dir, HOT_PERCENTILE, stats=stats
        )
        
        # Report results
        datasets_str = " + ".join(stats['datasets_available'])
        tqdm.write(f"{city_name}: Analysis complete ({datasets_str})")
        
        # Store summary
        analysis_results.append({
            'city': city_name,
            'status': 'success',
            'datasets': stats['datasets_available'],
            'stats_file': str(stats_file),
            'plot_file': str(plot_path) if plot_path else None
        })
        
    except Exception as e:
        tqdm.write(f"{city_name}: Analysis failed - {e}")
        analysis_results.append({
            'city': city_name,
            'status': 'failed',
            'error': str(e)
        })
        continue

# ====================================================================
# SUMMARY
# ====================================================================

print(f"\n{'='*80}")
print("HOT EVENT ANALYSIS COMPLETE")
print(f"{'='*80}")

# Categorize results
successful = [r for r in analysis_results if r['status'] == 'success']
skipped = [r for r in analysis_results if r['status'] == 'skipped']
failed = [r for r in analysis_results if r['status'] == 'failed']
no_data = [r for r in analysis_results if r['status'] == 'no_data']

print(f"\nSUMMARY:")
print(f"   Total cities: {len(analysis_results)}")
print(f"   Successful: {len(successful)}")
print(f"   Skipped (existing): {len(skipped)}")
print(f"   No data: {len(no_data)}")
print(f"   Failed: {len(failed)}")

if successful:
    print(f"\nSUCCESSFULLY ANALYZED CITIES:")
    for result in successful:
        datasets_str = " + ".join(result['datasets'])
        print(f"   • {result['city']}: {datasets_str}")

if skipped:
    print(f"\nSKIPPED CITIES (files exist):")
    for result in skipped:
        print(f"   • {result['city']}")

if failed:
    print(f"\nFAILED CITIES:")
    for result in failed:
        print(f"   • {result['city']}: {result.get('error', 'unknown error')}")

print(f"\nOUTPUT LOCATIONS:")
print(f"    Statistics (JSON): data/outputs/04_trend_analysis_daily/City_[Name]/Data/")
print(f"    Visualizations (PNG): data/outputs/04_trend_analysis_daily/City_[Name]/Figures/")

print(f"\n{'='*80}")
print("SECTION 3 COMPLETE")
print("Next: Review statistics and proceed to inter-dataset comparison analysis")
print(f"{'='*80}")

---
### 3.2 Enhanced Visualization with Persistent Heat Event Markers

Create enhanced plots showing persistent heat events with special markers connecting consecutive days/nights above threshold. This visualization highlights sustained heat periods more clearly.

In [ ]:
# ====================================================================
# ENHANCED VISUALIZATION WITH PERSISTENT EVENT MARKERS
# ====================================================================

from visualization import create_enhanced_persistent_event_plot

print("="*80)
print("ENHANCED VISUALIZATION WITH PERSISTENT EVENT MARKERS")
print("="*80)

for city_name in tqdm(list(city_geometries.keys()), desc="Enhanced plots", unit="city"):
    tqdm.write(f"\n{'='*80}")
    tqdm.write(f"Processing: {city_name}")
    tqdm.write(f"{'='*80}")
    
    # Define paths
    city_output_dir = config.OUTPUT_DIR / "04_trend_analysis_daily" / f"City_{city_name}" / "Data"
    city_plots_dir = config.OUTPUT_DIR / "04_trend_analysis_daily" / f"City_{city_name}" / "Figures"
    stats_file = city_output_dir / f"{city_name}_comprehensive_LST_statistics_{STUDY_YEAR}.json"
    
    try:
        # Load datasets
        tqdm.write(f"    Loading datasets...")
        modis_df, era5_df, data_status = load_dual_datasets(
            city_name, STUDY_YEAR, config.OUTPUT_DIR
        )
        
        if data_status == 'no_data':
            tqdm.write(f"    {city_name}: No data available")
            continue
        
        # Load statistics
        if stats_file.exists():
            tqdm.write(f"    Loading statistics...")
            import json
            with open(stats_file, 'r') as f:
                stats = json.load(f)
        else:
            tqdm.write(f"    Statistics file not found, skipping {city_name}")
            continue
        
        # Create enhanced visualization
        tqdm.write(f"    Creating enhanced visualization...")
        plot_path = create_enhanced_persistent_event_plot(
            modis_df, era5_df, city_name, STUDY_YEAR, city_plots_dir, HOT_PERCENTILE, stats=stats
        )
        
        tqdm.write(f"    ✅ {city_name}: Enhanced plot complete")
        
    except Exception as e:
        tqdm.write(f"    ❌ {city_name}: Failed - {e}")
        continue

print(f"\n{'='*80}")
print("ENHANCED VISUALIZATION COMPLETE")
print(f"{'='*80}")
print(f"\nOutput location: data/outputs/04_trend_analysis_daily/City_[Name]/Figures/")
print(f"File pattern: *_MODIS_ERA5_persistent_markers_{STUDY_YEAR}.png")
print(f"\nPersistent events are shown as:")
print(f"  • Thick horizontal lines with 'o' markers at start/end")
print(f"  • Positioned above threshold lines")
print(f"  • Day events: darkred (MODIS), darkorange (ERA5)")
print(f"  • Night events: darkblue (MODIS), indigo (ERA5)")
print(f"\n{'='*80}")

---

## SECTION 4: Comprehensive Summary Tables

Generate consolidated summary tables for cross-city comparison and analysis.

### Summary Tables Generated

1. **All Cities Summary CSV** - Complete metrics for all cities and datasets
2. **Dataset Comparison Table** - Side-by-side MODIS vs ERA5-Land comparison

### Metrics Included

**Data Coverage:**
- Day/night observation counts and percentages

**Temperature Statistics:**
- Mean, min, max temperatures for day and night
- Standard deviations

**Heat Events:**
- Hot event counts (day/night)
- Persistent event counts and maximum durations
- Threshold values

**Extreme Values:**
- Highest temperatures recorded
- Dates of extreme events

---

In [ ]:
# ====================================================================
# COMPREHENSIVE SUMMARY TABLES
# ====================================================================

import json
import glob

def create_all_cities_summary_csv(output_base_dir, study_year):
    """
    Create comprehensive CSV summary table from all city JSON statistics files
    """
    pattern = str(output_base_dir / "04_trend_analysis_daily" / "City_*" / "Data" / f"*_comprehensive_LST_statistics_{study_year}.json")
    json_files = glob.glob(pattern)
    
    summary_data = []
    
    for json_file in json_files:
        try:
            city_name = Path(json_file).stem.replace(f'_comprehensive_LST_statistics_{study_year}', '')
            
            with open(json_file, 'r') as f:
                stats = json.load(f)
            
            # Process each dataset (MODIS and/or ERA5-Land)
            for dataset in stats.get('datasets_available', []):
                dataset_key = dataset.lower().replace('-', '_').replace(' ', '_')
                
                if dataset_key not in stats:
                    continue
                
                ds = stats[dataset_key]
                
                # Extract persistent event details
                persist_days_list = ds['heat_events']['persistent_events']['persistent_hot_days']
                persist_nights_list = ds['heat_events']['persistent_events']['persistent_hot_nights']
                
                max_duration_days = max([e['duration'] for e in persist_days_list], default=0)
                max_duration_nights = max([e['duration'] for e in persist_nights_list], default=0)
                
                summary_row = {
                    'city': city_name,
                    'dataset': dataset,
                    'year': study_year,
                    
                    # Coverage
                    'total_days': ds['data_coverage']['total_days'],
                    'day_coverage_count': ds['data_coverage']['day_coverage_count'],
                    'day_coverage_pct': round(ds['data_coverage']['day_coverage_percent'], 1),
                    'night_coverage_count': ds['data_coverage']['night_coverage_count'],
                    'night_coverage_pct': round(ds['data_coverage']['night_coverage_percent'], 1),
                    
                    # Temperature statistics (using median)
                    'day_temp_mean': round(ds['temperature_statistics']['day']['median']['mean'], 2),
                    'day_temp_min': round(ds['temperature_statistics']['day']['median']['min'], 2),
                    'day_temp_max': round(ds['temperature_statistics']['day']['median']['max'], 2),
                    'day_temp_std': round(ds['temperature_statistics']['day']['median']['std'], 2),
                    
                    'night_temp_mean': round(ds['temperature_statistics']['night']['median']['mean'], 2),
                    'night_temp_min': round(ds['temperature_statistics']['night']['median']['min'], 2),
                    'night_temp_max': round(ds['temperature_statistics']['night']['median']['max'], 2),
                    'night_temp_std': round(ds['temperature_statistics']['night']['median']['std'], 2),
                    
                    # Heat events
                    'day_hot_threshold': round(ds['heat_events']['thresholds']['day_hot_threshold'], 2),
                    'night_hot_threshold': round(ds['heat_events']['thresholds']['night_hot_threshold'], 2),
                    'hot_days_count': ds['heat_events']['hot_events_count']['hot_days_count'],
                    'hot_nights_count': ds['heat_events']['hot_events_count']['hot_nights_count'],
                    
                    # Persistent events
                    'persistent_days_count': ds['heat_events']['persistent_events']['persistent_hot_days_count'],
                    'persistent_days_max_duration': max_duration_days,
                    'persistent_nights_count': ds['heat_events']['persistent_events']['persistent_hot_nights_count'],
                    'persistent_nights_max_duration': max_duration_nights,
                    
                    # Extremes
                    'highest_day_temp': round(ds['extreme_temperatures']['highest_day_median'], 2),
                    'highest_day_date': ds['extreme_temperatures']['date_highest_day'],
                    'highest_night_temp': round(ds['extreme_temperatures']['highest_night_median'], 2),
                    'highest_night_date': ds['extreme_temperatures']['date_highest_night']
                }
                
                summary_data.append(summary_row)
                
        except Exception as e:
            print(f"Error processing {json_file}: {e}")
    
    if not summary_data:
        print("No data found for summary table")
        return None
    
    # Create DataFrame and sort
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values(['city', 'dataset'])
    
    # Save to CSV
    summary_path = output_base_dir / "04_trend_analysis_daily" / f'all_cities_LST_summary_{study_year}.csv'
    summary_df.to_csv(summary_path, index=False)
    
    print(f"All cities summary saved: {summary_path}")
    print(f"    Records: {len(summary_df)} dataset-city combinations")
    print(f"    Cities: {summary_df['city'].nunique()} unique cities")
    print(f"    Datasets: {summary_df['dataset'].nunique()} unique datasets")
    
    return summary_path

def create_modis_era5_comparison_table(output_base_dir, study_year):
    """
    Create side-by-side comparison table for cities with both MODIS and ERA5-Land
    """
    pattern = str(output_base_dir / "04_trend_analysis_daily" / "City_*" / "Data" / f"*_comprehensive_LST_statistics_{study_year}.json")
    json_files = glob.glob(pattern)
    
    comparison_data = []
    
    for json_file in json_files:
        try:
            city_name = Path(json_file).stem.replace(f'_comprehensive_LST_statistics_{study_year}', '')
            
            with open(json_file, 'r') as f:
                stats = json.load(f)
            
            # Only process cities with both datasets
            if 'modis' not in stats or 'era5_land' not in stats:
                continue
            
            m = stats['modis']
            e = stats['era5_land']
            
            # Extract persistent event details
            m_persist_days = m['heat_events']['persistent_events']['persistent_hot_days']
            m_persist_nights = m['heat_events']['persistent_events']['persistent_hot_nights']
            e_persist_days = e['heat_events']['persistent_events']['persistent_hot_days']
            e_persist_nights = e['heat_events']['persistent_events']['persistent_hot_nights']
            
            comparison_row = {
                'city': city_name,
                'year': study_year,
                
                # Coverage comparison
                'modis_day_coverage_pct': round(m['data_coverage']['day_coverage_percent'], 1),
                'era5_day_coverage_pct': round(e['data_coverage']['day_coverage_percent'], 1),
                'modis_night_coverage_pct': round(m['data_coverage']['night_coverage_percent'], 1),
                'era5_night_coverage_pct': round(e['data_coverage']['night_coverage_percent'], 1),
                
                # Temperature comparison
                'modis_day_mean': round(m['temperature_statistics']['day']['median']['mean'], 2),
                'era5_day_mean': round(e['temperature_statistics']['day']['median']['mean'], 2),
                'day_temp_diff': round(e['temperature_statistics']['day']['median']['mean'] - m['temperature_statistics']['day']['median']['mean'], 2),
                
                'modis_night_mean': round(m['temperature_statistics']['night']['median']['mean'], 2),
                'era5_night_mean': round(e['temperature_statistics']['night']['median']['mean'], 2),
                'night_temp_diff': round(e['temperature_statistics']['night']['median']['mean'] - m['temperature_statistics']['night']['median']['mean'], 2),
                
                # Persistent events comparison
                'modis_persist_days': len(m_persist_days),
                'modis_persist_days_max': max([d['duration'] for d in m_persist_days], default=0),
                'era5_persist_days': len(e_persist_days),
                'era5_persist_days_max': max([d['duration'] for d in e_persist_days], default=0),
                
                'modis_persist_nights': len(m_persist_nights),
                'modis_persist_nights_max': max([n['duration'] for n in m_persist_nights], default=0),
                'era5_persist_nights': len(e_persist_nights),
                'era5_persist_nights_max': max([n['duration'] for n in e_persist_nights], default=0),
                
                # Extreme temperatures
                'modis_max_day': round(m['extreme_temperatures']['highest_day_median'], 2),
                'era5_max_day': round(e['extreme_temperatures']['highest_day_median'], 2),
                'modis_max_night': round(m['extreme_temperatures']['highest_night_median'], 2),
                'era5_max_night': round(e['extreme_temperatures']['highest_night_median'], 2)
            }
            
            comparison_data.append(comparison_row)
            
        except Exception as e:
            print(f"Error processing {json_file}: {e}")
    
    if not comparison_data:
        print("No cities with both datasets for comparison")
        return None
    
    # Create DataFrame and sort
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('city')
    
    # Save to CSV
    comparison_path = output_base_dir / "04_trend_analysis_daily" / f'MODIS_vs_ERA5Land_comparison_{study_year}.csv'
    comparison_df.to_csv(comparison_path, index=False)
    
    print(f"\nDataset comparison table saved: {comparison_path}")
    print(f"    Cities with both datasets: {len(comparison_df)}")
    
    return comparison_path

# ====================================================================
# EXECUTE SUMMARY GENERATION
# ====================================================================

print("="*80)
print("GENERATING COMPREHENSIVE SUMMARY TABLES")
print("="*80)

# Generate all cities summary
summary_csv = create_all_cities_summary_csv(config.OUTPUT_DIR, STUDY_YEAR)

# Generate MODIS vs ERA5-Land comparison
comparison_csv = create_modis_era5_comparison_table(config.OUTPUT_DIR, STUDY_YEAR)

print(f"\n{'='*80}")
print("SUMMARY TABLE GENERATION COMPLETE")
print(f"{'='*80}")

if summary_csv:
    print(f"\nAll Cities Summary: {summary_csv.name}")
if comparison_csv:
    print(f"MODIS vs ERA5 Comparison: {comparison_csv.name}")

print(f"\nFiles location: {config.OUTPUT_DIR / '04_trend_analysis_daily'}")
print(f"\n{'='*80}")
print("SECTION 4 COMPLETE")
print("All daily LST analysis outputs generated successfully")
print(f"{'='*80}")

<a id="section-4"></a>
---
## 4. Inter-Dataset Validation: MODIS vs ERA5-Land

**Purpose**: Quantify agreement between satellite observations (MODIS) and reanalysis model (ERA5-Land)

**Why This Matters**:
- **Validate ERA5-Land accuracy** against satellite truth
- **Identify systematic biases** (temperature offsets)
- **Assess correlation strength** (temporal pattern consistency)
- **Quantify prediction errors** (RMSE, MAE)
- **Build confidence** in using ERA5 for gap-filling when MODIS has cloud cover

**Metrics Calculated**:
1. **Pearson Correlation (r)**: Measures how well the two datasets track together
   - r > 0.9 = Excellent agreement
   - 0.7 < r < 0.9 = Good agreement
   - r < 0.7 = Poor agreement (investigate issues)

2. **Bias (°C)**: Mean difference between datasets
   - Positive = ERA5 warmer than MODIS
   - Negative = ERA5 cooler than MODIS
   - Near zero = No systematic offset

3. **RMSE (°C)**: Root Mean Square Error - overall prediction accuracy
   - Lower is better
   - Includes both bias and random errors

4. **MAE (°C)**: Mean Absolute Error - robust to outliers
   - Average magnitude of differences
   - Intuitive interpretation

**Expected Patterns**:
- High correlation (r > 0.85): ERA5 captures temporal variability well
- Small bias (< 2°C): ERA5 model is well-calibrated
- Moderate RMSE (2-4°C): Typical for reanalysis vs satellite comparison
- Day/night differences: Models often perform differently for day vs night

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ====================================================================
# SECTION 4: INTER-DATASET COMPARISON - MODIS vs ERA5-Land
# Pearson correlation, bias, RMSE, MAE for validation
# ====================================================================

from src.temporal_analysis import compare_datasets
import matplotlib.pyplot as plt

# Reload visualization module to get latest functions
import importlib
import src.visualization
importlib.reload(src.visualization)
from src.visualization import plot_multi_city_dataset_comparison

print("\n" + "="*80)
print("INTER-DATASET VALIDATION: MODIS vs ERA5-Land")
print("="*80)
print("\nComparing satellite observations (MODIS) with reanalysis model (ERA5-Land)")
print("This validates ERA5-Land accuracy and identifies systematic biases\n")

# Process each city
comparison_results = []
comparison_plot_data = []  # For multi-panel scatter plots

for city_name in tqdm(CITIES, desc="MODIS vs ERA5", unit="city"):
    tqdm.write(f"\n{'='*80}")
    tqdm.write(f"CITY: {city_name}")
    tqdm.write(f"{'='*80}")
    
    city_output_dir = config.OUTPUT_DIR / '04_trend_analysis_daily' / f'City_{city_name}'
    data_dir = city_output_dir / 'Data'
    
    # Check if both MODIS and ERA5 data exist
    modis_file = data_dir / f'{city_name}_raw_daily_MODIS_{STUDY_YEAR}.csv'
    era5_file = data_dir / f'{city_name}_raw_daily_ERA5_{STUDY_YEAR}.csv'
    
    has_modis = modis_file.exists()
    has_era5 = era5_file.exists()
    
    if not has_modis:
        tqdm.write(f"WARNING: MODIS data not found: {modis_file.name}")
        continue
    
    if not has_era5:
        tqdm.write(f"WARNING: ERA5-Land data not found: {era5_file.name}")
        continue
    
    # --- Skip logic: reuse cached comparison if outputs already exist ---
    comparison_csv = data_dir / f'{city_name}_MODIS_vs_ERA5_comparison_{STUDY_YEAR}.csv'
    json_path = data_dir / f'{city_name}_comparison_stats_{STUDY_YEAR}.json'
    
    if SKIP_EXISTING and comparison_csv.exists() and json_path.exists():
        tqdm.write(f"[SKIP] Cached comparison found – loading from disk")
        try:
            import json as _json
            with open(json_path, 'r') as f:
                cached_stats = _json.load(f)
            
            # Validate cached data has expected keys
            if all(k in cached_stats for k in ('city', 'daytime', 'nighttime')):
                comparison_results.append(cached_stats)
                
                # Also load merged CSV for scatter-plot data
                merged_comparison = pd.read_csv(comparison_csv)
                
                # Re-derive light-weight comparison dicts for the plot list
                has_day = cached_stats['daytime']['correlation'] is not None
                has_night = cached_stats['nighttime']['correlation'] is not None
                
                if has_day and has_night:
                    comparison_plot_data.append({
                        'city': city_name,
                        'merged_data': merged_comparison,
                        'day_comparison': {
                            'correlation': cached_stats['daytime']['correlation'],
                            'bias': cached_stats['daytime']['bias'],
                            'rmse': cached_stats['daytime']['rmse'],
                            'mae': cached_stats['daytime']['mae'],
                            'n_samples': cached_stats['daytime']['n_samples'],
                            'merged_data': merged_comparison
                        },
                        'night_comparison': {
                            'correlation': cached_stats['nighttime']['correlation'],
                            'bias': cached_stats['nighttime']['bias'],
                            'rmse': cached_stats['nighttime']['rmse'],
                            'mae': cached_stats['nighttime']['mae'],
                            'n_samples': cached_stats['nighttime']['n_samples'],
                            'merged_data': merged_comparison
                        }
                    })
                
                day_r = cached_stats['daytime']['correlation']
                night_r = cached_stats['nighttime']['correlation']
                day_str = f"r={day_r:.3f}" if day_r is not None else "N/A"
                night_str = f"r={night_r:.3f}" if night_r is not None else "N/A"
                tqdm.write(f"   ✅ Loaded: Day {day_str} | Night {night_str}")
                continue  # skip to next city
            else:
                tqdm.write(f"   ⚠️  Cached JSON has unexpected format – re-computing")
        except Exception as e:
            tqdm.write(f"   ⚠️  Failed to load cache ({e}) – re-computing")
    
    tqdm.write(f"[OK] Both datasets available - proceeding with comparison\n")
    
    # Load data
    modis_df = pd.read_csv(modis_file)
    era5_df = pd.read_csv(era5_file)
    
    # Perform comparison for DAY
    tqdm.write("-" * 80)
    tqdm.write("DAYTIME COMPARISON")
    tqdm.write("-" * 80)
    
    day_comparison = compare_datasets(
        df1=modis_df,
        df2=era5_df,
        temp_column1='day_lst_median',
        temp_column2='day_lst_median',
        dataset1_name='MODIS',
        dataset2_name='ERA5-Land'
    )
    
    if day_comparison and day_comparison.get('n_samples', 0) > 0 and day_comparison.get('correlation') is not None:
        tqdm.write(f"\nDaytime Statistics:")
        tqdm.write(f"   Correlation (r):     {day_comparison['correlation']:.3f}")
        tqdm.write(f"   Bias (MODIS-ERA5):   {day_comparison['bias']:+.2f}°C")
        tqdm.write(f"   RMSE:                {day_comparison['rmse']:.2f}°C")
        tqdm.write(f"   MAE:                 {day_comparison['mae']:.2f}°C")
        tqdm.write(f"   Sample size (n):     {day_comparison['n_samples']} days")
        
        # Interpretation
        r = day_comparison['correlation']
        if r > 0.9:
            tqdm.write(f"   --> Excellent agreement! ERA5 tracks MODIS very well")
        elif r > 0.7:
            tqdm.write(f"   --> Good agreement. ERA5 captures temporal patterns")
        else:
            tqdm.write(f"   WARNING: Moderate agreement. Review potential issues")
    else:
        tqdm.write("\n   No overlapping daytime data for comparison (0 matching dates or all NaN)")
    
    # Perform comparison for NIGHT
    tqdm.write("\n" + "-" * 80)
    tqdm.write("NIGHTTIME COMPARISON")
    tqdm.write("-" * 80)
    
    night_comparison = compare_datasets(
        df1=modis_df,
        df2=era5_df,
        temp_column1='night_lst_median',
        temp_column2='night_lst_median',
        dataset1_name='MODIS',
        dataset2_name='ERA5-Land'
    )
    
    if night_comparison and night_comparison.get('n_samples', 0) > 0 and night_comparison.get('correlation') is not None:
        tqdm.write(f"\nNighttime Statistics:")
        tqdm.write(f"   Correlation (r):     {night_comparison['correlation']:.3f}")
        tqdm.write(f"   Bias (MODIS-ERA5):   {night_comparison['bias']:+.2f}°C")
        tqdm.write(f"   RMSE:                {night_comparison['rmse']:.2f}°C")
        tqdm.write(f"   MAE:                 {night_comparison['mae']:.2f}°C")
        tqdm.write(f"   Sample size (n):     {night_comparison['n_samples']} days")
        
        # Interpretation
        r = night_comparison['correlation']
        if r > 0.9:
            tqdm.write(f"   --> Excellent agreement! ERA5 tracks MODIS very well")
        elif r > 0.7:
            tqdm.write(f"   --> Good agreement. ERA5 captures temporal patterns")
        else:
            tqdm.write(f"   WARNING: Moderate agreement. Review potential issues")
    else:
        tqdm.write("\n   No overlapping nighttime data for comparison (0 matching dates or all NaN)")
    
    # Save comparison data
    has_day = day_comparison and day_comparison.get('n_samples', 0) > 0 and day_comparison.get('correlation') is not None
    has_night = night_comparison and night_comparison.get('n_samples', 0) > 0 and night_comparison.get('correlation') is not None
    
    if has_day and has_night:
        tqdm.write(f"\nSaving comparison statistics...")
        
        # Get merged data for plotting
        merged_comparison = pd.merge(
            day_comparison['merged_data'],
            night_comparison['merged_data'],
            on='date',
            how='outer'
        )
        
        # Save CSV
        comparison_csv = data_dir / f'{city_name}_MODIS_vs_ERA5_comparison_{STUDY_YEAR}.csv'
        merged_comparison.to_csv(comparison_csv, index=False)
        tqdm.write(f"   Saved: {comparison_csv.name}")
        
        # Save JSON summary
        comparison_stats = {
            'city': city_name,
            'year': STUDY_YEAR,
            'daytime': {
                'correlation': day_comparison['correlation'],
                'bias': day_comparison['bias'],
                'rmse': day_comparison['rmse'],
                'mae': day_comparison['mae'],
                'n_samples': day_comparison['n_samples']
            },
            'nighttime': {
                'correlation': night_comparison['correlation'],
                'bias': night_comparison['bias'],
                'rmse': night_comparison['rmse'],
                'mae': night_comparison['mae'],
                'n_samples': night_comparison['n_samples']
            }
        }
        
        json_path = data_dir / f'{city_name}_comparison_stats_{STUDY_YEAR}.json'
        import json
        with open(json_path, 'w') as f:
            json.dump(comparison_stats, f, indent=2)
        tqdm.write(f"   Saved: {json_path.name}")
        
        # Store for multi-city comparison
        comparison_results.append(comparison_stats)
        comparison_plot_data.append({
            'city': city_name,
            'merged_data': merged_comparison,
            'day_comparison': day_comparison,
            'night_comparison': night_comparison
        })
    elif has_day or has_night:
        tqdm.write(f"\n   Partial comparison data available - saving what we have")
        comparison_stats = {
            'city': city_name,
            'year': STUDY_YEAR,
            'daytime': {
                'correlation': day_comparison['correlation'] if has_day else None,
                'bias': day_comparison['bias'] if has_day else None,
                'rmse': day_comparison['rmse'] if has_day else None,
                'mae': day_comparison['mae'] if has_day else None,
                'n_samples': day_comparison['n_samples'] if has_day else 0
            },
            'nighttime': {
                'correlation': night_comparison['correlation'] if has_night else None,
                'bias': night_comparison['bias'] if has_night else None,
                'rmse': night_comparison['rmse'] if has_night else None,
                'mae': night_comparison['mae'] if has_night else None,
                'n_samples': night_comparison['n_samples'] if has_night else 0
            }
        }
        comparison_results.append(comparison_stats)
    else:
        tqdm.write(f"\n   No valid comparison data for {city_name} - skipping")

# Summary
print(f"\n{'='*80}")
print(f"COMPARISON SUMMARY: {len(comparison_results)} of {len(CITIES)} cities compared")
print(f"{'='*80}")

if comparison_results:
    for stats in comparison_results:
        day_r = stats['daytime']['correlation']
        night_r = stats['nighttime']['correlation']
        day_str = f"r={day_r:.3f}" if day_r is not None else "N/A"
        night_str = f"r={night_r:.3f}" if night_r is not None else "N/A"
        print(f"  {stats['city']:20s}  Day: {day_str}  |  Night: {night_str}")
else:
    print("  No valid comparisons could be made")

# Multi-city scatter plot (always regenerated from loaded data)
if comparison_plot_data:
    print(f"\nCreating multi-city comparison scatter plots...")
    try:
        plot_dir = config.OUTPUT_DIR / '04_trend_analysis_daily'
        fig = plot_multi_city_dataset_comparison(
            comparison_plot_data,
            dataset1_name='MODIS',
            dataset2_name='ERA5-Land',
            output_dir=plot_dir
        )
        plt.show()
        plt.close()
    except Exception as e:
        print(f"Warning: Could not create multi-city plot: {e}")
else:
    print("\nNo comparison data available for plotting")

<a id="section-4a"></a>
---
## 4a. Target Diagram: Multi-City Performance Summary

**Purpose**: Visualize validation metrics for all cities simultaneously using a Target Diagram

**What is a Target Diagram?**
- Standard visualization in atmospheric science for model evaluation
- Shows **3 metrics in one plot**: Bias, RMSE, and Correlation
- Distance from center (bulls-eye) indicates agreement quality
- Closer to center = better agreement between datasets

**How to Read the Diagram:**
- **X-axis**: Unbiased RMSE (random error component)
- **Y-axis**: Bias (systematic error)
- **Point markers**: Each point represents one city-period comparison
  - Orange circles = Daytime comparisons
  - Grey circles = Nighttime comparisons
- **Bulls-eye (center)**: Perfect agreement (zero bias, zero error)
- **Distance from center**: Overall RMSE
- **Concentric circles**: RMSE performance tiers (2°C, 4°C, 6°C intervals)

**Interpretation Guidelines:**
- Points near **origin**: Excellent agreement (low bias + low random error)
- Points on **horizontal axis**: Unbiased but has random error
- Points far from **horizontal axis**: Systematic bias present
- **Positive Y-axis**: Dataset 1 (MODIS) warmer than Dataset 2 (ERA5-Land)
- **Negative Y-axis**: Dataset 1 (MODIS) cooler than Dataset 2 (ERA5-Land)

**What This Shows:**
- Which cities have best/worst agreement
- Whether errors are systematic (bias) or random (scatter)
- Day vs Night performance differences
- Overall pattern of ERA5-Land validation against MODIS
- Magnitude of total RMSE for each city-period combination

**Key Improvements in This Version:**
- Thinner reference circles (0.5 linewidth) for cleaner appearance
- 2°C spacing for better granularity at current city values
- Black city labels with adjustText library to prevent overlaps
- Bullseye target marker (concentric circles) at origin for perfect agreement
- Wider x-axis (140% of data range) providing more horizontal space
- All enhancements ensure scalability to 50+ cities without clutter

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ====================================================================
# SECTION 4a: TARGET DIAGRAM - Multi-City Validation Summary
# Visualize all validation metrics on a single standardized plot
# ====================================================================

# Reload visualization module to get latest functions
import importlib
import src.visualization
importlib.reload(src.visualization)
from src.visualization import plot_target_diagram

print("\n" + "="*80)
print("CREATING TARGET DIAGRAM")
print("="*80)
print("\nTarget Diagram shows bias, RMSE, and correlation for all cities")
print("simultaneously using a standardized coordinate system.\n")

if not comparison_results:
    print("WARNING: No comparison results available. Run Section 4 first.")
else:
    # Adapt nested dict format to flat format expected by plot_target_diagram
    # Nested: {'city':..., 'daytime': {'bias':..., 'rmse':..., 'correlation':...}, 'nighttime': {...}}
    # Flat:   {'city':..., 'day_bias':..., 'day_rmse':..., 'day_correlation':..., 'night_*':...}
    target_input = []
    for r in comparison_results:
        day = r.get('daytime', {})
        night = r.get('nighttime', {})
        # Skip cities missing either day or night data
        if day.get('correlation') is None or night.get('correlation') is None:
            print(f"  Skipping {r['city']} (incomplete comparison data)")
            continue
        target_input.append({
            'city': r['city'],
            'day_bias': day['bias'],
            'day_rmse': day['rmse'],
            'day_correlation': day['correlation'],
            'night_bias': night['bias'],
            'night_rmse': night['rmse'],
            'night_correlation': night['correlation'],
        })

    if not target_input:
        print("WARNING: No cities with complete day+night comparisons.")
    else:
        print(f"Processing {len(target_input)} cities for visualization...\n")
        
        # Create target diagram using modular function
        output_dir = config.OUTPUT_DIR / '04_trend_analysis_daily'
        
        fig, summary = plot_target_diagram(
            comparison_results=target_input,
            dataset1_name='MODIS',
            dataset2_name='ERA5-Land',
            output_dir=output_dir,
            figsize=None,  # Auto-size for publication (half A4)
            dpi=300,
            show_labels=True
        )
        
        # Display the plot
        plt.show()
        
        # Print summary statistics
        print(f"\n{'='*80}")
        print("TARGET DIAGRAM ANALYSIS")
        print(f"{'='*80}\n")
        
        print(f"Number of cities analyzed: {summary['n_cities']}")
        
        print(f"\n{'─'*80}")
        print("DAYTIME PERFORMANCE")
        print(f"{'─'*80}")
        print(f"  [BEST]  {summary['day']['best_city']:15s} (RMSE = {summary['day']['best_rmse']:.2f}°C)")
        print(f"  [WORST] {summary['day']['worst_city']:15s} (RMSE = {summary['day']['worst_rmse']:.2f}°C)")
        print(f"  Mean bias:      {summary['day']['mean_bias']:+.2f}°C")
        print(f"  Mean RMSE:      {summary['day']['mean_rmse']:.2f}°C")
        print(f"  Mean r:         {summary['day']['mean_correlation']:.3f}")
        
        print(f"\n{'─'*80}")
        print("NIGHTTIME PERFORMANCE")
        print(f"{'─'*80}")
        print(f"  [BEST]  {summary['night']['best_city']:15s} (RMSE = {summary['night']['best_rmse']:.2f}°C)")
        print(f"  [WORST] {summary['night']['worst_city']:15s} (RMSE = {summary['night']['worst_rmse']:.2f}°C)")
        print(f"  Mean bias:      {summary['night']['mean_bias']:+.2f}°C")
        print(f"  Mean RMSE:      {summary['night']['mean_rmse']:.2f}°C")
        print(f"  Mean r:         {summary['night']['mean_correlation']:.3f}")
        
        # Overall interpretation
        print(f"\n{'─'*80}")
        print("OVERALL BIAS PATTERN")
        print(f"{'─'*80}")
        bias_dir_day = 'warmer' if summary['day']['mean_bias'] < 0 else 'cooler'
        bias_dir_night = 'warmer' if summary['night']['mean_bias'] < 0 else 'cooler'
        print(f"  Day:   ERA5-Land is {bias_dir_day} than MODIS by {abs(summary['day']['mean_bias']):.2f}°C on average")
        print(f"  Night: ERA5-Land is {bias_dir_night} than MODIS by {abs(summary['night']['mean_bias']):.2f}°C on average")
        
        print(f"\n{'='*80}")
        print("KEY FINDINGS")
        print(f"{'='*80}")
        print("  - Target diagram consolidates all validation metrics in one plot")
        print("  - Distance from center (bullseye) indicates overall agreement quality")
        print("  - Y-axis shows systematic bias (can be corrected)")
        print("  - X-axis shows random error (cannot be easily corrected)")
        print("  - Concentric circles represent RMSE performance tiers (2, 4, 6°C)")
        print(f"  - Figure sized for publication (half A4 landscape = {fig.get_size_inches()[0]:.2f}\" × {fig.get_size_inches()[1]:.2f}\")")
        print("  - Black city labels with adjustText prevent overlaps (scalable to 50+ cities)")
        
        print(f"\n{'='*80}")
        print("TARGET DIAGRAM COMPLETE")
        print(f"{'='*80}")

# Section 5: Decision Tree for Event Selection

This section applies a **duration-priority decision tree** to select optimal heat events for high-resolution SEVIRI LST analysis. The decision tree prioritizes **longest ERA5-Land duration** (heat stress science) while documenting **MODIS validation quality** for transparency.

## Scientific Rationale

**Heat stress is cumulative** - longer events have greater public health and infrastructure impacts. Therefore, the **longest ERA5-Land event is always selected**, regardless of MODIS validation status. MODIS validation is documented for quality assessment but does not override duration priority.

## Decision Tree Logic (Option 1: Duration Paramount)

### Step 1: Select Longest ERA5-Land Event
- **Priority**: Maximum duration
- **Rationale**: Captures most severe cumulative heat stress
- **Always executed**: No alternatives considered after this step

### Step 2: Check MODIS Validation (Documentation Only)
- **Check 1**: Does MODIS **persistent event** overlap with ERA5?
  - YES → **Validation Level: Excellent** (strongest evidence)
  - NO → Proceed to Check 2
  
- **Check 2**: Does MODIS **hottest single day/night** overlap with ERA5?
  - YES → **Validation Level: Good** (moderate evidence)
  - NO → **Validation Level: Fair** (ERA5-only, no satellite validation)

### Step 3: Assign Quality Flag
- **Excellent**: Persistent MODIS event validates ERA5 selection
- **Good**: Hottest MODIS day/night validates ERA5 selection  
- **Fair**: No MODIS validation (cloud gaps, data limitations)

**Note**: Validation level is for documentation and stakeholder transparency. It does **NOT** affect event selection - longest ERA5 duration always wins.

## No ±1 Day Extensions

Previous logic extended MODIS events by ±1 day to find near-misses. This has been **removed** for simplicity:
- Direct overlap checks only
- Reduces complexity and potential errors
- Maintains focus on clear validation evidence

## Outputs Generated

1. **decision_tree_selections_2024.json**: Complete event selections with:
   - Selected event details (dates, duration, source)
   - Quality flag (Excellent/Good/Fair)
   - Validation level (excellent/good/none)
   - Decision stage (Duration_Priority_Selection)
   - Alternatives considered (other ERA5 events)

2. **decision_tree_summary_2024.csv**: Summary table for all cities (day/night)

3. **Console output**: Detailed decision process for each city showing duration and validation assessment

These selections will be used to determine which periods to analyze with high-resolution SEVIRI data (15-minute temporal resolution).

## Example: Port Harcourt Night

**Old Logic (MODIS-Priority)**: Selected 2-day event with MODIS validation  
**New Logic (Duration-Priority)**: Selected 12-day event (longest) → Fair quality (no MODIS, but longest duration)

**Result**: 500% longer event → Better heat stress characterization + 6x more SEVIRI timestamps (26 → 156) for gap filling

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ====================================================================
# SECTION 5: DECISION TREE FOR EVENT SELECTION
# Hierarchical approach prioritizing MODIS validation with ERA5-Land
# ====================================================================

from src import decision_tree_utils
import json
from pathlib import Path
import glob

print("="*70)
print("APPLYING DECISION TREE FOR EVENT SELECTION")
print("="*70)
print()
print("This analysis will:")
print("  1. Load comprehensive LST statistics for all cities")
print("  2. Apply 5-step decision tree (MODIS validation prioritized)")
print("  3. Select optimal events for SEVIRI high-resolution analysis")
print("  4. Generate decision logs and quality flags")
print("  5. Create summary tables showing selections")
print()

# Determine paths
dt_output_dir = str(project_root / 'data' / 'outputs' / '04_trend_analysis_daily')
results_file_path = Path(dt_output_dir) / f'decision_tree_selections_{STUDY_YEAR}.json'

# --- Skip logic: validate cached results against current input files ---
run_decision_tree = True
results = None
results_file = None

if SKIP_EXISTING and results_file_path.exists():
    print("[SKIP] Cached decision tree results found – validating...")
    try:
        with open(results_file_path, 'r') as f:
            cached_results = json.load(f)
        
        # Discover current stats input files (same glob as the utility function)
        pattern1 = str(Path(dt_output_dir) / "City_*" / "Data" / f"*_comprehensive_LST_statistics_{STUDY_YEAR}.json")
        pattern2 = str(Path(dt_output_dir) / "City_*" / "Daily_LST" / f"*_comprehensive_LST_statistics_{STUDY_YEAR}.json")
        current_files = glob.glob(pattern1) + glob.glob(pattern2)
        current_cities = sorted({
            Path(f).name.split('_comprehensive_LST_statistics_')[0]
            for f in current_files
        })
        cached_cities = sorted({r['city'] for r in cached_results})
        
        if current_cities == cached_cities:
            results = cached_results
            results_file = str(results_file_path)
            run_decision_tree = False
            print(f"   ✅ Cache valid – {len(results)} cities match current inputs")
        else:
            added = set(current_cities) - set(cached_cities)
            removed = set(cached_cities) - set(current_cities)
            if added:
                print(f"   ⚠️  New cities detected: {', '.join(sorted(added))}")
            if removed:
                print(f"   ⚠️  Removed cities: {', '.join(sorted(removed))}")
            print(f"   🔄 Re-running decision tree to match current inputs...")
    except Exception as e:
        print(f"   ⚠️  Failed to validate cache ({e}) – re-running")

if run_decision_tree:
    # Apply decision tree to all cities
    results, results_file = decision_tree_utils.process_all_cities_with_decision_tree(
        output_dir=dt_output_dir,
        study_year=STUDY_YEAR
    )

# Check if analysis was successful
if results is None or results_file is None:
    print()
    print("="*70)
    print("⚠️ DECISION TREE ANALYSIS FAILED")
    print("="*70)
    print()
    print("No statistics files found. Please ensure Section 3 has been run first.")
    print("Expected files: City_*/Data/*_comprehensive_LST_statistics_2024.json")
else:
    print()
    print("="*70)
    status_label = "LOADED FROM CACHE" if not run_decision_tree else "COMPLETE"
    print(f"DECISION TREE ANALYSIS {status_label}!")
    print("="*70)
    print()
    print(f"📊 Results Summary:")
    print(f"   Cities processed: {len(results)}")
    print(f"   Output file: {Path(results_file).name}")
    print(f"   Location: {Path(results_file).parent}")
    print()

    # Display summary of selected events
    print("Selected Events by City:")
    print("-" * 70)
    for result in results:
        city = result['city']
        print(f"\n{city}:")
        print(f"  Datasets available: {', '.join(result['datasets_available'])}")
        
        for period in ['day', 'night']:
            event = result['selected_events'][period]
            decision = result['decision_log'][period]
            
            if event:
                print(f"  {period.upper()}:")
                print(f"    Stage: {decision['final_decision_stage']}")
                print(f"    Quality: {decision['quality_flag']}")
                print(f"    Period: {event['period_used']['start_date']} to {event['period_used']['end_date']}")
                print(f"    Duration: {event['period_used']['duration']} days")
                print(f"    Source: {event['source']}")
            else:
                print(f"  {period.upper()}: No event selected ({decision['quality_flag']})")

    print()
    print("="*70)
    print("Ready for SEVIRI high-resolution analysis using selected periods!")
    print("="*70)

## 📡 **Section 6: SEVIRI High-Resolution LST Analysis**

### Overview
This section processes **SEVIRI LST data** at 15-minute temporal resolution and 3-km spatial resolution for the **selected persistent heat events** identified by the decision tree. SEVIRI (Spinning Enhanced Visible and InfraRed Imager) provides high-frequency thermal observations that complement coarser MODIS and ERA5-Land datasets.

---

### **Temporal Windows for LST Analysis**

Based on the selected events from the decision tree, SEVIRI data is extracted for specific time windows that capture peak thermal stress conditions:

#### **🌞 Daytime Window: 12:00 - 15:00 UTC**
- **Rationale**: Captures maximum LST during peak solar heating
- **Sampling**: 15-minute intervals = 13 timestamps per day
- **Scientific Basis**: 
  - Solar noon typically occurs 13:00-14:00 UTC in West Africa
  - Maximum surface temperatures observed 13:00-15:00 (Peng et al., 2012)
  - Urban heat island peaks in early-to-mid afternoon (Zhou et al., 2014; Chakraborty et al., 2019)
  - 3-hour window ensures sufficient temporal coverage for cloud gap filling

#### **🌙 Nighttime Window: 00:00 - 03:00 UTC**
- **Rationale**: Captures minimum LST during deepest nighttime cooling
- **Sampling**: 15-minute intervals = 13 timestamps per night
- **Scientific Basis**:
  - Minimum urban temperatures typically occur 02:00-04:00 (Peng et al., 2012)
  - Nocturnal urban heat island persists through early morning (Chakraborty & Lee, 2019)
  - 3-hour window provides robust temporal sampling for median composite computation
  - With longer event durations (6-18 days), extended windows maximize spatial coverage


---

### **Processing Workflow**

1. **Load Decision Tree Selections**
   - Retrieve selected DAY and NIGHT events for each city
   - Extract event date ranges and quality flags

2. **Generate Timestamp Lists**
   - For each event, create 15-minute timestamp list within specified windows
   - Example: 12-day event × 13 timestamps/day = 156 SEVIRI images

3. **Download SEVIRI LST Data**
   - Query EUMETSAT Data Store API for each timestamp
   - Download LST product at 3-km resolution
   - Automatically skip existing files

4. **Spatial Processing**
   - Clip SEVIRI data to city boundaries (AOI + 5km buffer)
   - Reproject to UTM coordinates
   - Create temporal median composite to reduce cloud contamination

5. **Visualization & Export**
   - Generate high-resolution LST maps
   - Export clipped rasters for further analysis
   - Calculate coverage statistics (% valid pixels)

---

### **Key References**

- **Peng, S., et al. (2012)**. Surface urban heat island across 419 global big cities. *Environmental Science & Technology*, 46(2), 696-703. [DOI: 10.1021/es2030438](https://doi.org/10.1021/es2030438)

- **Zhou, D., et al. (2014)**. Surface urban heat island in China's 32 major cities: Spatial patterns and drivers. *Remote Sensing of Environment*, 152, 51-61. [DOI: 10.1016/j.rse.2014.05.017](https://doi.org/10.1016/j.rse.2014.05.017)

- **Chakraborty, T., & Lee, X. (2019)**. A simplified urban-extent algorithm to characterize surface urban heat islands on a global scale and examine vegetation control on their spatiotemporal variability. *International Journal of Applied Earth Observation and Geoinformation*, 74, 269-280. [DOI: 10.1016/j.jag.2018.09.015](https://doi.org/10.1016/j.jag.2018.09.015)

- **Chakraborty, T., et al. (2019)**. A simplified approach for quantifying the proportion of the urban tree cover and its influence on the land surface temperature. *International Journal of Applied Earth Observation and Geoinformation*, 77, 158-169. [DOI: 10.1016/j.jag.2018.11.009](https://doi.org/10.1016/j.jag.2018.11.009)


[↑ Back to Table of Contents](#Table-of-Contents)


In [ ]:
# ====================================================================
# SECTION 6: PROCESS SEVIRI LST FOR SELECTED EVENTS
# High temporal resolution (15-min to daily) for decision tree periods
# ====================================================================

from src import seviri_processing_utils

print("="*70)
print("SEVIRI LST PROCESSING - HIGH TEMPORAL RESOLUTION")
print("="*70)
print()

# Check EUMETSAT availability
if not EUMETSAT_AVAILABLE:
    print("⚠️  EUMETSAT credentials not available")
    print("    SEVIRI processing requires EUMETSAT Data Store access")
    print("    Please configure secrets/eumetsat_credentials.json")
    print()
else:
    print("✓ EUMETSAT credentials loaded")
    print()
    
    # Configuration
    SEVIRI_CONFIG = {
        'temp_dir': str(project_root / 'data' / 'outputs' / '04_trend_analysis_daily' / 'temp_seviri'),
        'n_workers': 2,  # DASK workers for parallel downloads
    }
    
    print("Configuration:")
    print(f"  Temporary directory: {SEVIRI_CONFIG['temp_dir']}")
    print(f"  DASK workers: {SEVIRI_CONFIG['n_workers']}")
    print(f"  Cleanup policy: Downloaded ZIP files will be deleted automatically")
    print()
    
    # Create temp directory
    import os
    os.makedirs(SEVIRI_CONFIG['temp_dir'], exist_ok=True)
    
    # Process all cities with selected events
    print("="*70)
    print("Starting Multi-City SEVIRI Processing")
    print("="*70)
    print()
    print("This will:")
    print("  1. Load decision tree event selections")
    print("  2. Initialize DASK distributed computing")
    print("  3. Download SEVIRI data for selected periods (parallel)")
    print("  4. Compute median LST composites")
    print("  5. Reproject from MSG native to WGS84 @ 1km")
    print("  6. Clip to city AOI boundaries")
    print("  7. Delete temporary ZIP files (automatic cleanup)")
    print("  8. Generate metadata and processing summaries")
    print()
    print("Note: Skip logic will prevent reprocessing existing files")
    print()
    
    # Run processing
    summary = seviri_processing_utils.process_all_cities_seviri(
        eumetsat_creds=eumetsat_creds,  # From cell 7
        output_dir=str(project_root / 'data' / 'outputs'),
        study_year=STUDY_YEAR,
        temp_dir=SEVIRI_CONFIG['temp_dir'],
        aoi_dir=str(aoi_base_dir),
        city_list=None,  # Auto-detect from decision tree results
        n_workers=SEVIRI_CONFIG['n_workers']
    )
    
    # Display results
    if summary:
        print()
        print("="*70)
        print("SEVIRI PROCESSING COMPLETE!")
        print("="*70)
        print()
        print(f" Final Summary:")
        print(f"   Total cities: {summary['total_cities']}")
        print(f"   ✓ Successful: {len(summary['successful'])}")
        print(f"   ⊘ Skipped (already processed): {len(summary['skipped'])}")
        print(f"   ✗ Failed: {len(summary['failed'])}")
        print()
        
        if summary['successful']:
            print("Successful cities:")
            for city in summary['successful']:
                print(f"  - {city}")
        
        if summary['skipped']:
            print()
            print("Skipped cities (files already exist):")
            for city in summary['skipped']:
                print(f"  - {city}")
        
        if summary['failed']:
            print()
            print("Failed cities:")
            for city in summary['failed']:
                print(f"  - {city}")
        
        print()
        print(f"Summary saved: processing_summary_{STUDY_YEAR}.json")
        print()
        print("Output files per city:")
        print("  • Native MSG GeoTIFF (3km, geostationary projection)")
        print("  • Reprojected GeoTIFF (1km, WGS84)")
        print("  • Clipped GeoTIFF (1km, WGS84, city AOI)")
        print("  • Metadata JSON (processing details)")
        print()
        print("="*70)
        print("Ready for high-resolution temporal analysis!")
        print("="*70)
    else:
        print()
        print("="*70)
        print("⚠️  SEVIRI PROCESSING FAILED")
        print("="*70)
        print("Check console output above for error details")

<a id="seviri-viz"></a>
## 6.2 Visualize SEVIRI LST Results

**Purpose**: Create multi-city visualizations and summary statistics for processed SEVIRI LST data

**What This Section Does**:
- Automatically detects all processed cities from output directory
- Loads clipped SEVIRI GeoTIFFs and extracts spatial statistics
- Creates **two separate maps** (day and night) showing LST during persistent heat events
- Generates comprehensive summary statistics table with event durations
- Saves both PNG and PDF versions of all figures

**Output**:
- `Persistent_Heat_Event_Map_day_{year}.png/pdf`: Daytime LST map with geospatial features
- `Persistent_Heat_Event_Map_night_{year}.png/pdf`: Nighttime LST map with geospatial features
- `LST_statistics_summary_{year}.csv`: Statistical summary table with event details

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ====================================================================
# VISUALIZE SEVIRI LST - PERSISTENT HEAT EVENT MAPS
# Publication-ready multi-panel maps with all geospatial layers
# ====================================================================

from src.data_io import load_seviri_cities_lst, create_seviri_temperature_summary
from src.visualization import create_persistent_heat_event_maps

print("\n" + "=" * 60)
print("PERSISTENT HEAT EVENT MAPS")
print("=" * 60)

# Define SEVIRI output directory
seviri_output_dir = project_root / 'data' / 'outputs' / '04_trend_analysis_daily' / 'SEVIRI_LST'

# --- Skip logic: check if combined map files already exist ---
day_map_png = seviri_output_dir / f'Persistent_Heat_Event_Map_day_{STUDY_YEAR}.png'
night_map_png = seviri_output_dir / f'Persistent_Heat_Event_Map_night_{STUDY_YEAR}.png'

skip_maps = (
    SKIP_EXISTING
    and day_map_png.exists()
    and night_map_png.exists()
)

if skip_maps:
    print(f"\n[SKIP] Combined Persistent Heat Event Maps already exist:")
    print(f"   ✅ {day_map_png.name}")
    print(f"   ✅ {night_map_png.name}")
    print("   Set SKIP_EXISTING = False to force regeneration.")
    
    # Still load cities data for the summary table (lightweight)
    cities_data = load_seviri_cities_lst(
        str(seviri_output_dir),
        STUDY_YEAR
    )
    n_cities = len(cities_data)
    
    if n_cities > 0:
        print(f"\n   Loaded {n_cities} cities for summary table")
        for city_name, city_data in sorted(cities_data.items()):
            day_status = "✓" if 'day' in city_data else "✗"
            night_status = "✓" if 'night' in city_data else "✗"
            print(f"     {city_name}: Day {day_status} | Night {night_status}")
        
        # Summary table always regenerated (lightweight)
        print("\nRegenerating temperature statistics summary...")
        summary_df = create_seviri_temperature_summary(
            cities_data,
            base_dir=str(seviri_output_dir),
            study_year=STUDY_YEAR
        )
    
    print("\n" + "=" * 60)
    print("SEVIRI HEAT EVENT MAP GENERATION SKIPPED (cached)")
    print("=" * 60)

else:
    # Load all processed cities
    cities_data = load_seviri_cities_lst(
        str(seviri_output_dir),
        STUDY_YEAR
    )
    
    n_cities = len(cities_data)
    print(f"\nFound {n_cities} cities with processed LST data")
    
    if n_cities > 0:
        # Display city status
        for city_name, city_data in sorted(cities_data.items()):
            day_status = "✓" if 'day' in city_data else "✗"
            night_status = "✓" if 'night' in city_data else "✗"
            print(f"  {city_name}: Day {day_status} | Night {night_status}")
        
        print("\n" + "=" * 60)
        print("GENERATING PUBLICATION MAPS")
        print("=" * 60)
        print("\nCreating multi-panel Persistent Heat Event Maps with:")
        print("  • LST median during longest heat events")
        print("  • Water bodies (≥3 km²) as blue-filled polygons")
        print("  • Roads as thin grey lines")
        print("  • Places labeled (~1 per 5 km²)")
        print("  • Main/Adjacent boundaries as hollow polygons")
        print("  • Dynamic layout for any number of cities")
        print("  • Scale bars inside map frames")
        print("  • Unified legend and north arrow")
        
        # Create daytime Persistent Heat Event Map
        print("\n" + "-" * 60)
        print("DAYTIME PERSISTENT HEAT EVENT MAP")
        print("-" * 60)
        fig_day = create_persistent_heat_event_maps(
            cities_data=cities_data,
            period='day',
            base_dir=str(seviri_output_dir),
            study_year=STUDY_YEAR,
            aoi_dir=str(aoi_base_dir)
        )
        
        if fig_day:
            display(fig_day)
            plt.close(fig_day)
        
        # Create nighttime Persistent Heat Event Map
        print("\n" + "-" * 60)
        print("NIGHTTIME PERSISTENT HEAT EVENT MAP")
        print("-" * 60)
        fig_night = create_persistent_heat_event_maps(
            cities_data=cities_data,
            period='night',
            base_dir=str(seviri_output_dir),
            study_year=STUDY_YEAR,
            aoi_dir=str(aoi_base_dir)
        )
        
        if fig_night:
            display(fig_night)
            plt.close(fig_night)
        
        print("\n" + "=" * 60)
        print("MAPS CREATED SUCCESSFULLY")
        print("=" * 60)
        
        # Create summary statistics table
        print("\nGenerating temperature statistics summary...")
        summary_df = create_seviri_temperature_summary(
            cities_data,
            base_dir=str(seviri_output_dir),
            study_year=STUDY_YEAR
        )
        
        print("\n" + "=" * 60)
        print("SEVIRI LST ANALYSIS COMPLETE")
        print("=" * 60)
        print(f"Output location: {seviri_output_dir}")
        print("\n✅ Persistent Heat Event Maps saved:")
        print(f"   • Persistent_Heat_Event_Map_day_{STUDY_YEAR}.png/pdf")
        print(f"   • Persistent_Heat_Event_Map_night_{STUDY_YEAR}.png/pdf")
        
    else:
        print("\n  No processed SEVIRI data found.")
        print("     Run Section 6.1 (SEVIRI Processing) first.")

<a id="cross-city-analysis"></a>
## 📊 Cross-City Heat Event Analysis

This section provides comparative analytics across all cities to identify heat extremes and temperature stability patterns.

[↑ Back to Table of Contents](#Table-of-Contents)

---

<a id="summary-chart"></a>
### 7.1 Summary Chart: Mean LST Comparison

**Purpose**: Identify cities with above-average heat during persistent events

**Visualization Features**:
- **Grouped bars**: Day (left, orange) and Night (right, grey) for each city
- **Red tips**: Portions exceeding overall mean LST (danger zone)
- **Black range indicators**: Show min-max temperature variability (error bars with caps)
- **Dashed lines**: Overall mean LST (day = orange, night = grey)
- **Sorting**: Cities ordered from hottest to coolest (by combined day+night mean)

**Interpretation**:
- Cities with red tips experienced more extreme heat than the regional average
- Larger red portions = greater deviation from normal conditions
- Range indicators show temperature consistency within each city's heat event

---

<a id="variability-chart"></a>
### 7.2 Variability Chart: Temperature Stability Analysis

**Purpose**: Flag cities with unstable temperature conditions (high standard deviation)

**Visualization Features**:
- **Grouped bars**: Day (left, orange) and Night (right, grey) for each city
- **Red tips**: SD values exceeding overall mean SD (high variability zones)
- **Base colors**: Normal variability (SD ≤ Mean SD)
- **Dashed lines**: Overall mean SD thresholds (day = orange, night = grey)
- **Sorting**: Cities ordered by **largest daytime SD** (most variable first)

**Interpretation**:
- Red tips indicate cities with unpredictable temperature swings during heat events
- High variability may stress urban cooling systems and complicate heat adaptation
- Stable cities (no red tips) have more consistent temperatures during heat episodes


[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ====================================================================
# CROSS-CITY COMPARATIVE CHARTS
# ====================================================================

print("\n" + "=" * 80)
print("GENERATING CROSS-CITY ANALYSIS CHARTS")
print("=" * 80)

# Chart 1: Mean LST Comparison (identifies hottest cities)
print("\n[1/2] Summary Chart: Mean LST Comparison")
print("-" * 80)
fig_summary = create_persistent_heat_event_summary_chart(
    cities_data=cities_data,
    base_dir=seviri_output_dir,
    study_year=STUDY_YEAR,
    dpi=300
)

if fig_summary:
    display(fig_summary)
    plt.close(fig_summary)
else:
    print("⚠️  Could not generate summary chart - no valid data")

# Chart 2: Variability Analysis (identifies temperature instability)
print("\n[2/2] Variability Chart: Temperature Stability Analysis")
print("-" * 80)
fig_variability = create_persistent_heat_event_variability_chart(
    cities_data=cities_data,
    base_dir=seviri_output_dir,
    study_year=STUDY_YEAR,
    dpi=300
)

if fig_variability:
    display(fig_variability)
    plt.close(fig_variability)
else:
    print("⚠️  Could not generate variability chart - no valid data")

print("\n" + "=" * 80)
print("CROSS-CITY ANALYSIS COMPLETE")
print("=" * 80)
print(f"Output location: {seviri_output_dir}")
print("\n✅ Charts saved:")
print(f"   • Persistent_Heat_Event_Summary_{STUDY_YEAR}.png/pdf")
print(f"   • Persistent_Heat_Event_Variability_{STUDY_YEAR}.png/pdf")
print("\n📊 Key Insights:")
print("   • Summary chart: Identifies cities exceeding regional mean heat")
print("   • Variability chart: Flags cities with unstable temperature patterns")

<a id="combined-dot-plot"></a>
### 7.3 Paired Dot Plot: All Metrics in One Chart (Option A)

> **Module**: `src/visualization.py` → `create_paired_dot_plot()`

**Purpose**: Consolidate mean, spread, and exceedance into a single figure

**Visualization Features**:
- **Two markers per city**: Day (diamond) and Night (circle) connected by a thin line
- **Whiskers**: +/-1 SD from the mean (shows temperature stability)
- **Filled markers**: Cities exceeding overall mean (danger zone)
- **Hollow markers**: Cities below overall mean
- **Vertical dashed lines**: Overall day and night mean LST reference
- **Single sort order**: Combined day+night mean, hottest to coolest
- **2-layer donut inset**: Inner ring (dotted) = Day|Night totals; Outer ring = Danger|Normal counts
- **Danger-only annotations**: Mean LST values shown only for above-average cities

[Back to Table of Contents](#Table-of-Contents)


In [ ]:
# Option A: Paired Dot Plot (modularised → src/visualization.py)
import importlib, visualization
importlib.reload(visualization)
from visualization import create_paired_dot_plot

fig = create_paired_dot_plot(
    cities_data,
    base_dir=seviri_output_dir,
    study_year=STUDY_YEAR,
)
display(fig)
plt.close(fig)


<a id="scatter-pie"></a>
### 7.4 Day-Night Scatter with Heat Duration Pie Charts (Option C)

> **Module**: `src/visualization.py` → `create_daynight_scatter_pie()`

**Purpose**: Reveal day-night correlation, persistent heat exposure, and identify danger-zone cities

**Visualization Features**:
- **X-axis**: Day Mean LST | **Y-axis**: Night Mean LST
- **Pie charts** at each city position showing Day vs Night heat duration split
- **Pie size**: Total number of persistent heat days + nights (larger = longer heat exposure)
- **Orange slices**: Daytime persistent heat duration | **Dark grey**: Nighttime
- **Quadrant shading**: Top-right = danger zone, bottom-left = below average
- **Labels**: City names on transparent overlay (zorder=100), displaced by `adjustText`, with leader lines
- **Legend**: Bottom-centre, 2 columns, showing pie colours and size scale

**Quadrant Interpretation**:
| Quadrant | Day | Night | Meaning |
|----------|-----|-------|---------|
| Top-right | Hot | Hot | **Danger zone** – sustained heat day and night |
| Top-left | Cool | Hot | Night heat retention (possible UHI) |
| Bottom-right | Hot | Cool | Daytime heat stress, night relief |
| Bottom-left | Cool | Cool | Below-average heat events |

[Back to Table of Contents](#Table-of-Contents)


In [ ]:
# Option C: Day-Night Scatter with Pie Charts (modularised → src/visualization.py)
import importlib, visualization
importlib.reload(visualization)
from visualization import create_daynight_scatter_pie

fig = create_daynight_scatter_pie(
    cities_data,
    base_dir=seviri_output_dir,
    study_year=STUDY_YEAR,
)
display(fig)
plt.close(fig)


<a id="summary"></a>
##  Analysis Summary & Next Steps

[↑ Back to Table of Contents](#Table-of-Contents)

---

### Key Accomplishments

This notebook successfully completed **daily-scale LST temporal analysis** across Nigerian cities:

#### ✅ Data Processing
- **SEVIRI LST extraction**: Downloaded and processed MSG satellite data via OpenDAP
- **Persistent heat events**: Identified multi-day extreme heat episodes (≥2 consecutive days)
- **Water masking**: Filtered out water bodies ≥3 km² for accurate urban analysis
- **Multi-city coverage**: Processed 50 Nigerian cities via auto-discovery from Notebook 01 outputs

#### ✅ Geospatial Visualization
- **Heat event maps**: Created day/night LST maps with clipped boundaries
- **Layered features**: Integrated water bodies, roads, place labels, and scale bars
- **Publication-ready**: High-resolution PNG (300 DPI) and vector PDF outputs

- **Publication-ready combined charts**: Option A (Paired Dot Plot with donut inset) and Option C (Day-Night Scatter with Pie Charts) via `create_paired_dot_plot()` and `create_daynight_scatter_pie()` in `src/visualization.py`

#### ✅ Comparative Analytics
- **Summary chart**: Identified cities with above-average heat (red-tipped bars)
- **Variability chart**: Flagged cities with unstable temperatures (high SD, sorted by daytime SD)
- **Sorting & ranking**: Ordered cities by heat intensity and temperature stability
- **Statistical outputs**: Exported mean, SD, min, max values to CSV files

---

### Key Findings

#### Temperature Patterns
- Cities exhibit distinct heat profiles during persistent events
- Day-night temperature differences reveal diurnal cycle variations
- Some cities show consistent heat (low SD), others show volatile patterns (high SD)

#### Data Quality
- SEVIRI provides continuous monitoring at 15-minute intervals
- Water masking prevents bias from large rivers/lakes
- Geospatial features enable contextual interpretation

#### Regional Insights
- Cities exceeding mean LST (red tips in summary chart) face elevated heat stress
- High-variability cities (red tips in SD chart) have unpredictable cooling patterns
- Combined analysis identifies both **extreme heat** and **temperature instability**

---

### Output Files Generated

All results saved to: `data/outputs/04_trend_analysis_daily/SEVIRI_LST/`


### Next Steps: Hotspot Analysis (Notebook 05)

Now that we've identified **temporal patterns** (when heat events occur) and **city-level averages**, the next analysis focuses on **intra-city spatial patterns**:

#### 🔥 Notebook 05: Hotspot Analysis


**Analytical Methods**:
- **Getis-Ord Gi\* statistic**: Identify statistically significant hotspots and coldspots
- **Moran's I**: Measure global spatial autocorrelation
- **Local spatial clustering**: Detect neighborhood-scale heat patterns
- **Land cover overlay**: Integrate OpenStreetMap and JRC Global Surface Water

**Expected Outputs**:
- Hotspot significance maps (z-scores, p-values)
- Heat cluster classifications (high-high, low-low, outliers)
- Priority zones for heat adaptation interventions
- Spatial statistics summary tables

**Why This Matters**:
- City averages (Notebook 04) mask important within-city variability
- Hotspot analysis enables **neighborhood-targeted interventions**
- Identifies areas where vulnerable populations face greatest heat exposure
- Guides urban planning for green space, reflective surfaces, cooling centers

---
